# SentinelWard: audit, M1 baseline and M2 preparation

Historical working notebook, archived 15 October 2026. Code cells are preserved; outputs and runtime metadata are removed.

**Do not use Run all.** This interactive notebook includes download recovery cells and run-specific paths. Cells 1–9 audit data; 10–14 split, train and evaluate M1; 15–18 prepare VTaC for M2. Two extra cells after Cell 9 diagnose/recover downloads.

M1 test results have already been inspected. Do not retune against that test set. M2 has not been trained. Continue with the separate M2 notebook and saved prepared data; do not recreate the audit run. See ../docs/REPRODUCIBILITY.md.


In [ ]:
# CELL 1 — connect Drive and record the environment (CPU runtime is enough).
from pathlib import Path, PurePosixPath
from datetime import datetime, timezone
from collections import Counter, defaultdict
import csv, hashlib, importlib.metadata, io, json, platform, shutil, sys, zipfile

try:
    from google.colab import drive
except ImportError:
    raise RuntimeError("Open this notebook in Google Colab.")
drive.mount("/content/drive")

# Add a shortcut to the supplied DATA folder in My Drive and name it SentinelWard-data.
# If you use a different shortcut name, change only DATA_ROOT.
DATA_ROOT = Path("/content/drive/MyDrive/SentinelWard-data")
WORK_ROOT = Path("/content/drive/MyDrive/SentinelWard-work")
assert DATA_ROOT.is_dir(), f"Data folder not visible: {DATA_ROOT}. Check account and shortcut."
assert DATA_ROOT.resolve() != WORK_ROOT.resolve()
assert DATA_ROOT.resolve() not in WORK_ROOT.resolve().parents, "Keep outputs outside source data."
RUN_DIR = WORK_ROOT / "data-audit" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
RUN_DIR.mkdir(parents=True, exist_ok=False)

packages = {}
for name in ("numpy", "scipy", "pandas", "scikit-learn", "wfdb", "torch", "snapml"):
    try:
        packages[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        packages[name] = "not installed (not needed for this notebook)"
environment = {
    "python": sys.version, "platform": platform.platform(), "byteorder": sys.byteorder,
    "packages": packages, "local_free_GiB": round(shutil.disk_usage("/content").free / 2**30, 2),
    "data_root": str(DATA_ROOT), "purpose": "read-only data exploration; no model training"
}
(RUN_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(json.dumps(environment, indent=2))
print("\nVisible folders:", sorted(p.name for p in DATA_ROOT.iterdir() if p.is_dir()))
print("Reports:", RUN_DIR)


In [ ]:
# CELL 2 — inspect all four ZIP directories, without extracting waveforms.
# These exact file sizes were observed in Drive on 13 October 2026.
EXPECTED = {
    "vtac-1.1": ("vtac-a-benchmark-dataset-of-ventricular-tachycardia-alarms-from-icu-monitors-1.1.zip", 11695857538),
    "challenge-2015-1.0.0": ("reducing-false-arrhythmia-alarms-in-the-icu-the-physionetcomputing-in-cardiology-challenge-2015-1.0.0.zip", 369541924),
    "challenge-2011-1.0.0": ("improving-the-quality-of-ecgs-collected-using-mobile-phones-the-physionetcomputing-in-cardiology-challenge-2011-1.0.0.zip", 177891351),
    "nstdb-1.0.0": ("mit-bih-noise-stress-test-database-1.0.0.zip", 70945297),
}
ARCHIVES, inventory, problems = {}, [], []
for dataset, (filename, expected_bytes) in EXPECTED.items():
    path = DATA_ROOT / dataset / filename
    if not path.is_file():
        problems.append(f"{dataset}: missing {path}")
        continue
    try:
        with zipfile.ZipFile(path) as archive:
            members = [m for m in archive.infolist() if not m.is_dir()]
            names = [m.filename for m in members]
            unsafe = [n for n in names if PurePosixPath(n).is_absolute()
                      or ".." in PurePosixPath(n).parts or "\\" in n or ":" in n]
            duplicates = [n for n, count in Counter(names).items() if count > 1]
            if unsafe or duplicates:
                raise ValueError(f"Unsafe or duplicate member names: {(unsafe + duplicates)[:5]}")
            with (RUN_DIR / f"{dataset}_members.csv").open("w", newline="", encoding="utf-8") as out:
                writer = csv.writer(out)
                writer.writerow(["member", "uncompressed_bytes", "compressed_bytes", "zip_crc32"])
                writer.writerows((m.filename, m.file_size, m.compress_size, f"{m.CRC:08x}") for m in members)
            row = {
                "dataset": dataset, "archive_bytes": path.stat().st_size,
                "size_matches_drive": path.stat().st_size == expected_bytes,
                "files": len(members), "uncompressed_GiB": round(sum(m.file_size for m in members) / 2**30, 3),
                "extensions": dict(Counter(PurePosixPath(n).suffix.lower() for n in names)),
                "full_crc_verified": False, "publisher_checksums_verified": False
            }
            inventory.append(row)
            ARCHIVES[dataset] = path
            print(json.dumps(row, indent=2))
            if not row["size_matches_drive"]:
                problems.append(f"{dataset}: byte size differs from the observed Drive copy")
    except (OSError, zipfile.BadZipFile, ValueError) as error:
        problems.append(f"{dataset}: {error}")
(RUN_DIR / "archive_inventory.json").write_text(json.dumps(inventory, indent=2), encoding="utf-8")
(RUN_DIR / "inventory_problems.json").write_text(json.dumps(problems, indent=2), encoding="utf-8")
assert not problems, "\n".join(problems)
print("\nInventory complete. _parts was intentionally excluded; it is not another dataset.")
print("IMPORTANT: ZIP directories and sizes are checked; waveform integrity is still pending.")


In [ ]:
# CELL 3 — inspect metadata and audit the official VTaC split; never create a random split.
def read_small_text(archive, member, max_bytes=20_000_000):
    info = archive.getinfo(member)
    assert info.file_size <= max_bytes, f"Metadata unexpectedly large: {member}"
    return archive.read(member).decode("utf-8-sig")

def read_unique_csv(archive, basename):
    hits = [n for n in archive.namelist() if PurePosixPath(n).name == basename]
    assert len(hits) == 1, f"Expected one {basename}; found {hits}"
    content = read_small_text(archive, hits[0])
    (RUN_DIR / basename).write_text(content, encoding="utf-8")
    reader = csv.DictReader(io.StringIO(content))
    print(basename, "columns:", reader.fieldnames)
    assert reader.fieldnames and len(reader.fieldnames) == len(set(reader.fieldnames))
    rows = list(reader)
    assert rows and all(None not in r and all(v is not None for v in r.values()) for r in rows)
    print("Rows:", len(rows), "| First row:", rows[0])
    return rows

with zipfile.ZipFile(ARCHIVES["vtac-1.1"]) as archive:
    labels = read_unique_csv(archive, "event_labels.csv")
    splits = read_unique_csv(archive, "benchmark_data_split.csv")
    annotated_headers = [n for n in archive.namelist()
                         if "annotated_waveforms" in PurePosixPath(n).parts and n.endswith(".hea")]
    print("Annotated waveform headers:", len(annotated_headers))
    print("First annotated header:", annotated_headers[0] if annotated_headers else "NONE")
    if annotated_headers:
        print(read_small_text(archive, annotated_headers[0], max_bytes=100_000))
    checksum_files = [n for n in archive.namelist() if PurePosixPath(n).name == "SHA256SUMS.txt"]
    print("Publisher checksum manifests:", checksum_files)

# Read only bounded header/label previews for the other datasets.
for dataset, path in ARCHIVES.items():
    if dataset == "vtac-1.1":
        continue
    with zipfile.ZipFile(path) as archive:
        candidates = sorted(n for n in archive.namelist()
                            if n.endswith(".hea") or PurePosixPath(n).name.lower()
                            in {"alarms", "alarms.txt", "records", "reference.csv", "set-a.csv", "set-b.csv"})
        print("\n", dataset, "header/label candidates:", candidates[:12])
        headers = [n for n in candidates if n.endswith(".hea")]
        if headers:
            print(read_small_text(archive, headers[0], max_bytes=100_000)[:2000])

# Stop for review if actual metadata differs from this documented schema.
assert {"record", "event", "decision"} <= labels[0].keys(), "Review printed label columns."
assert {"record", "event"} <= splits[0].keys(), "Review printed split columns."
split_columns = [c for c in splits[0] if c.lower().strip() in {"split", "set"}]
assert len(split_columns) == 1, "Review printed split columns; do not guess the split field."
split_column = split_columns[0]
aliases = {"train": "train", "training": "train", "val": "validation",
           "valid": "validation", "validation": "validation", "test": "test", "testing": "test"}
def event_key(row):
    key = (row["record"].strip(), row["event"].strip())
    assert all(key), "Missing record/event identity."
    return key
label_keys, split_keys = [event_key(r) for r in labels], [event_key(r) for r in splits]
assert len(label_keys) == len(set(label_keys)), "Duplicate label event."
assert len(split_keys) == len(set(split_keys)), "Duplicate split event."
assert set(label_keys) == set(split_keys), "Labels and split keys do not match."
assert all(r["decision"].strip() for r in labels), "Missing label decision."
label_map = {event_key(r): r["decision"].strip() for r in labels}
groups, counts, decisions = defaultdict(set), Counter(), defaultdict(Counter)
for row in splits:
    value = row[split_column].strip().lower()
    assert value in aliases, f"Unknown split value: {value!r}"
    split = aliases[value]
    key = event_key(row)
    groups[key[0]].add(split)
    counts[split] += 1
    decisions[split][label_map[key]] += 1
overlap = {record: sorted(parts) for record, parts in groups.items() if len(parts) > 1}
assert not overlap, f"Record leakage across official splits: {list(overlap.items())[:10]}"
assert set(counts) == {"train", "validation", "test"}, "Missing split."
expected_counts = {"train": 4060, "validation": 495, "test": 482}
audit = {
    "events": len(labels), "record_ids": len(groups), "events_per_split": dict(counts),
    "decisions_per_split": {s: dict(c) for s, c in decisions.items()},
    "record_ids_per_split": {s: sum(s in v for v in groups.values()) for s in counts},
    "record_overlap": overlap, "matches_team_doc11_event_counts": dict(counts) == expected_counts,
    "status": "metadata audited; waveform checks and patient-identity semantics pending",
    "note": "Disjoint record IDs alone do not prove disjoint people across records or datasets."
}
(RUN_DIR / "vtac_split_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
print("\n", json.dumps(audit, indent=2))
assert dict(counts) == expected_counts, "Counts differ from doc 11: investigate before proceeding."
print("\nNEXT: send archive inventory and this split audit for review.")
print("Do not train yet. Full checksums, waveform-label joins, missing signals and label encoding remain to verify.")


In [ ]:
# CELL 4 — inspect nested training data and print compact results.
from pathlib import Path, PurePosixPath
from collections import Counter
import json, shutil, tempfile, zipfile

assert "ARCHIVES" in globals(), "Run Cells 1 and 2 first."

for dataset in ("challenge-2015-1.0.0", "challenge-2011-1.0.0"):
    print(f"\n=== {dataset} ===")

    with zipfile.ZipFile(ARCHIVES[dataset]) as outer:
        members = [m for m in outer.infolist() if not m.is_dir()]
        print("Outer file count:", len(members))

        # Print all names compactly so the 2011 structure is visible.
        print("Outer filenames:")
        for member in members:
            print(f"  {member.filename} ({member.file_size:,} bytes)")

        if dataset != "challenge-2015-1.0.0":
            continue

        matches = [
            m for m in members
            if PurePosixPath(m.filename).name.lower() == "training.zip"
        ]
        assert len(matches) == 1, (
            f"Expected exactly one training.zip; found {len(matches)}"
        )
        training = matches[0]

        # Stage only this nested archive on Colab's temporary disk.
        # Streaming avoids loading the entire archive into RAM.
        free = shutil.disk_usage("/content").free
        assert free > training.file_size + 512 * 1024**2, (
            "Insufficient local disk space for the nested archive."
        )

        with tempfile.TemporaryDirectory(
            prefix="sentinelward-inspect-", dir="/content"
        ) as temporary:
            local_zip = Path(temporary) / "training.zip"
            with outer.open(training) as source, local_zip.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)

            assert local_zip.stat().st_size == training.file_size

            with zipfile.ZipFile(local_zip) as inner:
                files = [m for m in inner.infolist() if not m.is_dir()]
                headers = [
                    m.filename for m in files
                    if m.filename.lower().endswith(".hea")
                ]
                label_candidates = [
                    m.filename for m in files
                    if any(term in PurePosixPath(m.filename).name.lower()
                           for term in ("alarm", "reference", "label"))
                ]

                print("\nINSIDE training.zip")
                print("Files:", len(files))
                print("Extensions:", dict(Counter(
                    PurePosixPath(m.filename).suffix.lower() or "(none)"
                    for m in files
                )))
                print("Header count:", len(headers))
                print("First 8 filenames:", [m.filename for m in files[:8]])
                print("Label candidates:", label_candidates)

                # Preview only small candidate label files.
                for name in label_candidates[:3]:
                    info = inner.getinfo(name)
                    if info.file_size <= 2_000_000:
                        text = inner.read(name).decode("utf-8-sig")
                        print(f"\nFirst 8 lines of {name}:")
                        print("\n".join(text.splitlines()[:8]))

print("\nInspection complete. No source files changed.")

In [ ]:
# CELL 5 — verify 2015 labels/files and inspect 2011 Set A.
import csv
import io
import json
import shutil
import tempfile
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath

assert "ARCHIVES" in globals() and "RUN_DIR" in globals(), (
    "Run Cells 1 and 2 first."
)

tasks = [
    ("challenge-2015-1.0.0", "training.zip"),
    ("challenge-2011-1.0.0", "Android-set-a.zip"),
]
verification = {}

for dataset, nested_name in tasks:
    print(f"\n=== {dataset}: {nested_name} ===", flush=True)

    with zipfile.ZipFile(ARCHIVES[dataset]) as outer:
        matches = [
            m for m in outer.infolist()
            if not m.is_dir()
            and PurePosixPath(m.filename).name == nested_name
        ]
        assert len(matches) == 1, f"Expected one {nested_name}: {matches}"
        member = matches[0]

        assert shutil.disk_usage("/content").free > (
            member.file_size + 512 * 1024**2
        ), "Insufficient temporary disk space."

        with tempfile.TemporaryDirectory(
            prefix="sentinelward-check-", dir="/content"
        ) as temporary:
            local_path = Path(temporary) / nested_name
            with outer.open(member) as source, local_path.open("wb") as target:
                shutil.copyfileobj(source, target, length=8 * 1024**2)

            assert local_path.stat().st_size == member.file_size

            with zipfile.ZipFile(local_path) as inner:
                files = [m for m in inner.infolist() if not m.is_dir()]
                names = [m.filename for m in files]
                assert len(names) == len(set(names)), "Duplicate archive paths."

                print("Checking all nested-file CRCs...", flush=True)
                bad_file = inner.testzip()
                assert bad_file is None, f"CRC failure: {bad_file}"
                print("Nested archive CRC: PASS")

                if dataset == "challenge-2015-1.0.0":
                    alarm_paths = [
                        n for n in names
                        if PurePosixPath(n).name == "ALARMS"
                    ]
                    assert len(alarm_paths) == 1, "Ambiguous ALARMS file."
                    text = inner.read(alarm_paths[0]).decode("utf-8-sig")
                    rows = [
                        [value.strip() for value in row]
                        for row in csv.reader(io.StringIO(text))
                        if row and any(value.strip() for value in row)
                    ]
                    assert all(len(row) == 3 for row in rows)
                    assert all(row[2] in {"0", "1"} for row in rows)

                    ids = [row[0] for row in rows]
                    headers = [
                        PurePosixPath(n).stem
                        for n in names if n.lower().endswith(".hea")
                    ]
                    waveforms = [
                        PurePosixPath(n).stem
                        for n in names if n.lower().endswith(".mat")
                    ]

                    for label, values in (
                        ("labels", ids), ("headers", headers),
                        ("waveforms", waveforms)
                    ):
                        assert len(values) == len(set(values)), (
                            f"Duplicate record IDs in {label}"
                        )

                    assert set(ids) == set(headers) == set(waveforms), (
                        "Label/header/waveform IDs do not match."
                    )
                    counts = Counter(row[2] for row in rows)
                    assert len(rows) == 750, f"Unexpected count: {len(rows)}"
                    assert counts == Counter({"0": 456, "1": 294}), (
                        f"Counts differ from team data notes: {counts}"
                    )

                    manifest = RUN_DIR / "challenge2015_labels.csv"
                    with manifest.open(
                        "w", newline="", encoding="utf-8"
                    ) as output:
                        writer = csv.writer(output)
                        writer.writerow(["record_id", "alarm_type", "label"])
                        writer.writerows(rows)

                    report = {
                        "nested_crc_pass": True,
                        "records": len(rows),
                        "label_counts": dict(counts),
                        "alarm_type_counts": dict(Counter(r[1] for r in rows)),
                        "label_header_waveform_match": True,
                    }
                    print(json.dumps(report, indent=2))
                    print("Label manifest saved:", manifest)

                else:
                    # Inspect file names before choosing a label parser.
                    candidates = [
                        m for m in files
                        if any(word in PurePosixPath(m.filename).name.lower()
                               for word in (
                                   "readme", "record", "label", "reference",
                                   "answer", "quality", "set-a"
                               ))
                    ]
                    report = {
                        "nested_crc_pass": True,
                        "files": len(files),
                        "extensions": dict(Counter(
                            PurePosixPath(n).suffix.lower() or "(none)"
                            for n in names
                        )),
                        "first_12_paths": names[:12],
                        "metadata_candidates": [
                            {"path": m.filename, "bytes": m.file_size}
                            for m in candidates[:20]
                        ],
                        "non_csv_paths_first_20": [
                            n for n in names
                            if not n.lower().endswith(".csv")
                        ][:20],
                    }
                    print(json.dumps(report, indent=2))

                    for item in candidates[:5]:
                        if item.file_size <= 200_000:
                            raw = inner.read(item.filename)
                            try:
                                preview = raw.decode("utf-8-sig")
                            except UnicodeDecodeError:
                                continue
                            print(f"\nPreview: {item.filename}")
                            print("\n".join(preview.splitlines()[:8])[:2000])

                verification[dataset] = report

result_path = RUN_DIR / "nested_archive_verification.json"
result_path.write_text(
    json.dumps(verification, indent=2), encoding="utf-8"
)
print("\nSaved:", result_path)
print("CRC checks detect corruption; publisher-checksum verification is separate.")

In [ ]:
# CELL 6 — identify 2011 labels and signal metadata; no signal decoding yet.
from pathlib import Path, PurePosixPath
from collections import Counter
import json, shutil, zipfile
import xml.etree.ElementTree as ET

assert "ARCHIVES" in globals() and "RUN_DIR" in globals()

CACHE = Path("/content/sentinelward-cache")
CACHE.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ARCHIVES["challenge-2011-1.0.0"]) as outer:
    matches = [
        m for m in outer.infolist()
        if PurePosixPath(m.filename).name == "Android-set-a.zip"
    ]
    assert len(matches) == 1, "Expected exactly one Android-set-a.zip."
    member = matches[0]

    # CRC in the filename distinguishes this cached archive version.
    cached = CACHE / f"Android-set-a-{member.CRC:08x}.zip"
    if not cached.exists() or cached.stat().st_size != member.file_size:
        assert shutil.disk_usage(CACHE).free > member.file_size + 100_000_000
        partial = cached.with_suffix(".partial")
        with outer.open(member) as source, partial.open("wb") as target:
            shutil.copyfileobj(source, target, length=8 * 1024**2)
        assert partial.stat().st_size == member.file_size
        partial.replace(cached)

SET_A_ZIP = cached  # Reuse this in later cells.

with zipfile.ZipFile(SET_A_ZIP) as archive:
    files = [m for m in archive.infolist() if not m.is_dir()]
    xml_files = sorted(
        m.filename for m in files if m.filename.lower().endswith(".xml")
    )
    raw_files = sorted(
        m.filename for m in files if m.filename.lower().endswith(".raw")
    )
    txt_files = sorted(
        m.filename for m in files if m.filename.lower().endswith(".txt")
    )

    xml_ids = [PurePosixPath(n).stem for n in xml_files]
    raw_ids = [PurePosixPath(n).stem for n in raw_files]
    pairing = {
        "xml_count": len(xml_ids),
        "raw_count": len(raw_ids),
        "duplicate_xml_ids": [
            k for k, v in Counter(xml_ids).items() if v > 1
        ],
        "duplicate_raw_ids": [
            k for k, v in Counter(raw_ids).items() if v > 1
        ],
        "xml_without_raw": sorted(set(xml_ids) - set(raw_ids)),
        "raw_without_xml": sorted(set(raw_ids) - set(xml_ids)),
        "raw_file_sizes_bytes": dict(Counter(
            archive.getinfo(n).file_size for n in raw_files
        )),
    }
    print("PAIRING AND RAW FILE SIZES")
    print(json.dumps(pairing, indent=2))

    print("\nTEXT FILE PREVIEWS")
    text_previews = {}
    for name in txt_files:
        info = archive.getinfo(name)
        assert info.file_size <= 2_000_000, f"Unexpectedly large text: {name}"
        text = archive.read(name).decode("utf-8-sig")
        lines = text.splitlines()
        preview = [line[:180] for line in lines[:4]]
        text_previews[name] = {"lines": len(lines), "preview": preview}
        print(f"\n{name} — {len(lines)} lines")
        print("\n".join(preview))

    assert xml_files, "No XML files found."
    example = xml_files[0]
    assert archive.getinfo(example).file_size <= 2_000_000
    root = ET.fromstring(archive.read(example))

    print(f"\nXML METADATA EXAMPLE: {example}")
    print("Root:", root.tag)
    print("Tag counts:", dict(Counter(
        node.tag.split("}")[-1] for node in root.iter()
    )))

    # Bounded structural preview; do not assume a label field or byte order.
    xml_preview = []
    for node in list(root.iter())[:35]:
        entry = {
            "tag": node.tag.split("}")[-1],
            "attributes": {
                key: str(value)[:160] for key, value in node.attrib.items()
            },
            "text": (node.text or "").strip()[:160],
        }
        xml_preview.append(entry)
        print(json.dumps(entry))

report = {
    "pairing": pairing,
    "text_previews": text_previews,
    "xml_example": example,
    "xml_preview": xml_preview,
}
report_path = RUN_DIR / "challenge2011_format_inspection.json"
report_path.write_text(json.dumps(report, indent=2), encoding="utf-8")
print("\nSaved:", report_path)
print("Local cached archive:", SET_A_ZIP)

In [ ]:
# CELL 7 — attach official quality labels to the 2011 records.
from pathlib import PurePosixPath
from collections import Counter
from urllib.request import Request, urlopen
from datetime import datetime, timezone
import csv, hashlib, json, re, zipfile

assert "SET_A_ZIP" in globals() and SET_A_ZIP.is_file(), (
    "Run Cell 6 first to recreate the local Set A cache."
)
assert "RUN_DIR" in globals()

BASE = "https://physionet.org/files/challenge-2011/1.0.0/set-a/"
label_sets, provenance = {}, {}

for quality in ("acceptable", "unacceptable"):
    filename = f"RECORDS-{quality}"
    url = BASE + filename
    request = Request(url, headers={"User-Agent": "SentinelWard-data-audit/1.0"})

    with urlopen(request, timeout=60) as response:
        payload = response.read(1_000_001)

    assert len(payload) <= 1_000_000, "Unexpectedly large label response."
    ids = [
        line.strip()
        for line in payload.decode("utf-8-sig").splitlines()
        if line.strip()
    ]
    assert ids and all(re.fullmatch(r"\d+", value) for value in ids), (
        f"{filename}: expected numeric record IDs, not HTML or another format."
    )
    assert len(ids) == len(set(ids)), f"Duplicate IDs in {filename}."

    label_sets[quality] = set(ids)
    (RUN_DIR / filename).write_bytes(payload)
    provenance[quality] = {
        "url": url,
        "sha256": hashlib.sha256(payload).hexdigest(),
        "records": len(ids),
        "retrieved_utc": datetime.now(timezone.utc).isoformat(),
    }

assert not (
    label_sets["acceptable"] & label_sets["unacceptable"]
), "Conflicting official labels."

with zipfile.ZipFile(SET_A_ZIP) as archive:
    raw_members = [
        name for name in archive.namelist()
        if name.lower().endswith(".raw")
    ]

    raw_by_id = {}
    for name in raw_members:
        stem = PurePosixPath(name).stem
        assert re.fullmatch(r"ecg\d+", stem), f"Unexpected filename: {name}"
        record_id = stem[3:]  # ecg1002603 -> 1002603
        assert record_id not in raw_by_id, f"Duplicate record: {record_id}"
        raw_by_id[record_id] = name

official_ids = label_sets["acceptable"] | label_sets["unacceptable"]
assert official_ids <= set(raw_by_id), (
    f"Official labels missing from archive: {sorted(official_ids - set(raw_by_id))}"
)

rows = []
for record_id in sorted(raw_by_id):
    if record_id in label_sets["acceptable"]:
        quality, target = "acceptable", 0
    elif record_id in label_sets["unacceptable"]:
        quality, target = "unacceptable", 1
    else:
        quality, target = "unlabelled", ""

    rows.append({
        "record_id": record_id,
        "quality": quality,
        "target_unacceptable": target,
        "raw_archive_path": raw_by_id[record_id],
    })

counts = dict(Counter(row["quality"] for row in rows))
assert counts == {
    "acceptable": 773, "unacceptable": 225, "unlabelled": 2
}, f"Unexpected label counts; investigate: {counts}"

manifest = RUN_DIR / "challenge2011_quality_labels.csv"
with manifest.open("w", newline="", encoding="utf-8") as output:
    writer = csv.DictWriter(output, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

(RUN_DIR / "challenge2011_label_provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)

print("QUALITY LABEL COUNTS:", counts)
print("Unlabelled IDs:", [
    row["record_id"] for row in rows if row["quality"] == "unlabelled"
])
print("Target definition: 1 = unacceptable; 0 = acceptable.")
print("Unlabelled targets remain blank, never converted to 0.")
print("Saved:", manifest)

In [ ]:
# CELL 8 — load two official WFDB examples and inspect physical signals.
%pip -q install wfdb==4.3.1

from pathlib import Path
from urllib.request import Request, urlopen
import csv, hashlib, importlib.metadata, json
import numpy as np
import matplotlib.pyplot as plt
import wfdb

assert "RUN_DIR" in globals(), "Run the earlier setup cells first."

manifest = RUN_DIR / "challenge2011_quality_labels.csv"
with manifest.open(encoding="utf-8", newline="") as source:
    label_rows = list(csv.DictReader(source))

# Fixed examples for checking the loader, not selecting model settings.
examples = [
    sorted(
        (r for r in label_rows if r["quality"] == quality),
        key=lambda r: r["record_id"]
    )[0]
    for quality in ("acceptable", "unacceptable")
]

BASE = "https://physionet.org/files/challenge-2011/1.0.0/set-a/"
LOCAL = Path("/content/sentinelward-cache/2011-wfdb-examples")
LOCAL.mkdir(parents=True, exist_ok=True)
file_provenance, checks = {}, []

def fetch_file(filename):
    assert Path(filename).name == filename
    assert "/" not in filename and "\\" not in filename
    request = Request(
        BASE + filename,
        headers={"User-Agent": "SentinelWard-data-audit/1.0"}
    )
    with urlopen(request, timeout=60) as response:
        payload = response.read(2_000_001)
    assert 0 < len(payload) <= 2_000_000, (
        f"Unexpected file size: {filename}"
    )
    (LOCAL / filename).write_bytes(payload)
    file_provenance[filename] = {
        "url": BASE + filename,
        "bytes": len(payload),
        "sha256": hashlib.sha256(payload).hexdigest(),
    }

for example in examples:
    record_id = example["record_id"]
    fetch_file(record_id + ".hea")
    header = wfdb.rdheader(str(LOCAL / record_id))

    assert header.record_name == record_id
    assert header.n_sig == 12, f"Unexpected channel count: {header.n_sig}"
    assert header.fs == 500, f"Unexpected sampling rate: {header.fs}"

    # Read filenames from the header rather than guessing the data layout.
    for filename in sorted(set(header.file_name)):
        fetch_file(filename)

    record = wfdb.rdrecord(str(LOCAL / record_id), physical=True)
    signals = record.p_signal
    assert signals.shape == (record.sig_len, record.n_sig)
    assert record.sig_len / record.fs >= 10

    channel_checks = []
    for index, name in enumerate(record.sig_name):
        values = signals[:, index]
        finite_values = values[np.isfinite(values)]
        channel_checks.append({
            "lead": name,
            "unit": record.units[index],
            "nonfinite_samples": int((~np.isfinite(values)).sum()),
            "min": float(finite_values.min()) if finite_values.size else None,
            "max": float(finite_values.max()) if finite_values.size else None,
            "constant_among_finite_samples": (
                bool(np.ptp(finite_values) == 0)
                if finite_values.size else None
            ),
        })

    summary = {
        "record_id": record_id,
        "whole_record_quality": example["quality"],
        "shape_samples_by_channels": list(signals.shape),
        "sampling_rate_hz": float(record.fs),
        "duration_seconds": float(record.sig_len / record.fs),
        "channels": channel_checks,
    }
    checks.append(summary)
    print(json.dumps(summary, indent=2))

    # Plot every lead: the label applies to the complete 12-lead recording.
    time_s = np.arange(record.sig_len) / record.fs
    fig, axes = plt.subplots(6, 2, figsize=(14, 12), sharex=True)
    for index, axis in enumerate(axes.flat):
        axis.plot(time_s, signals[:, index], linewidth=0.65)
        axis.set_title(record.sig_name[index], fontsize=10)
        axis.set_ylabel(record.units[index])
        axis.grid(alpha=0.2)
    axes[-1, 0].set_xlabel("Time (seconds)")
    axes[-1, 1].set_xlabel("Time (seconds)")
    fig.suptitle(
        f"{record_id} — whole-record label: {example['quality']}\n"
        "Original physical signals; each subplot has its own amplitude scale",
        fontsize=12
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(
        RUN_DIR / f"challenge2011_{record_id}_raw.png",
        dpi=140, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

report = {
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("wfdb", "numpy", "matplotlib")
    },
    "files": file_provenance,
    "records": checks,
    "scope": "Two-record loader check only; full dataset not yet decoded.",
}
(RUN_DIR / "challenge2011_waveform_examples.json").write_text(
    json.dumps(report, indent=2, allow_nan=False), encoding="utf-8"
)
print("\nSaved waveform checks and both plots to:", RUN_DIR)

In [ ]:
# CELL 9 — prepare the full 2011 WFDB corpus and audit before splitting.
from pathlib import Path, PurePosixPath
from urllib.request import Request, urlopen
from collections import Counter
import csv, hashlib, json, shutil, tarfile
import numpy as np
import wfdb

assert "RUN_DIR" in globals(), "Run the earlier setup first."

CACHE = Path("/content/sentinelward-cache")
CACHE.mkdir(parents=True, exist_ok=True)
TAR_PATH = CACHE / "challenge2011-set-a.tar.gz"
WFDB_ROOT = CACHE / "challenge2011-set-a-wfdb"
WFDB_ROOT.mkdir(parents=True, exist_ok=True)
URL = "https://physionet.org/files/challenge-2011/1.0.0/set-a.tar.gz"

# Download atomically: interrupted downloads never become the cached archive.
if not TAR_PATH.exists():
    assert shutil.disk_usage(CACHE).free > 600_000_000, (
        "Need at least 600 MB free for download and staging."
    )
    partial = TAR_PATH.with_suffix(".partial")
    request = Request(URL, headers={"User-Agent": "SentinelWard-audit/1.0"})
    with urlopen(request, timeout=120) as response, partial.open("wb") as out:
        expected = response.headers.get("Content-Length")
        received, last_report = 0, 0
        while True:
            block = response.read(4 * 1024**2)
            if not block:
                break
            out.write(block)
            received += len(block)
            if received - last_report >= 20 * 1024**2:
                print(f"Downloaded {received / 1024**2:.1f} MiB", flush=True)
                last_report = received
    if expected is not None:
        assert received == int(expected), "Incomplete download."
    assert received > 0
    partial.replace(TAR_PATH)

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024**2), b""):
            digest.update(block)
    return digest.hexdigest()

# Copy only regular WFDB files; do not use unrestricted extractall().
seen = set()
with tarfile.open(TAR_PATH, "r:gz") as archive:
    for member in archive:
        name = PurePosixPath(member.name)
        if not member.isfile() or name.suffix.lower() not in {".hea", ".dat"}:
            continue
        assert not name.is_absolute() and ".." not in name.parts
        assert name.stem.isdigit(), f"Unexpected record filename: {member.name}"
        assert name.name not in seen, f"Duplicate filename: {name.name}"
        seen.add(name.name)
        assert member.size <= 10_000_000, "Unexpectedly large record file."
        source = archive.extractfile(member)
        assert source is not None
        with source, (WFDB_ROOT / name.name).open("wb") as target:
            shutil.copyfileobj(source, target)

with (RUN_DIR / "challenge2011_quality_labels.csv").open(
    encoding="utf-8", newline=""
) as stream:
    labels = list(csv.DictReader(stream))

assert len(labels) == 1000
assert len({r["record_id"] for r in labels}) == 1000

# Match against this archive's members, not any older cached files.
wanted = {r["record_id"] for r in labels}
header_ids = {Path(n).stem for n in seen if n.endswith(".hea")}
data_ids = {Path(n).stem for n in seen if n.endswith(".dat")}
assert wanted == header_ids == data_ids, "Archive/label IDs do not match."

audit_rows = []
for number, row in enumerate(labels, start=1):
    record_id = row["record_id"]
    record_path = str(WFDB_ROOT / record_id)
    header = wfdb.rdheader(record_path)
    assert header.record_name == record_id
    assert header.n_sig == 12 and header.fs == 500
    assert header.sig_len == 5000, f"Unexpected duration: {record_id}"
    assert set(header.file_name) == {record_id + ".dat"}
    assert all(unit == "mV" for unit in header.units)

    digital = wfdb.rdrecord(record_path, physical=False).d_signal
    physical = wfdb.rdrecord(record_path, physical=True).p_signal
    assert digital.shape == physical.shape == (5000, 12)

    # WFDB checksums compare the low 16 bits of each channel's sample sum.
    assert all(value is not None for value in header.checksum)
    actual = np.sum(digital.astype(np.int64), axis=0) & 0xFFFF
    expected = np.asarray(header.checksum, dtype=np.int64) & 0xFFFF
    assert np.array_equal(actual, expected), (
        f"Digital checksum mismatch: {record_id}"
    )

    # Include decoding metadata with the data hash when grouping duplicates.
    # Matching groups mean identical encoded data and interpretation.
    signature = {
        "dat_sha256": sha256_file(WFDB_ROOT / f"{record_id}.dat"),
        "fs": header.fs,
        "samples": header.sig_len,
        "leads": header.sig_name,
        "units": header.units,
        "format": header.fmt,
        "gain": header.adc_gain,
        "baseline": header.baseline,
    }
    group_hash = hashlib.sha256(
        json.dumps(signature, sort_keys=True).encode()
    ).hexdigest()

    audit_rows.append({
        "record_id": record_id,
        "quality": row["quality"],
        "target_unacceptable": row["target_unacceptable"],
        "waveform_group": group_hash,
        "nonfinite_samples": int((~np.isfinite(physical)).sum()),
        "constant_channels": int(np.sum(np.all(
            digital == digital[0:1, :], axis=0
        ))),
        "digital_checksum_pass": True,
        "previously_inspected": record_id in {"1002867", "1002603"},
    })

    if number % 100 == 0:
        print(f"Verified {number}/1000 records", flush=True)

manifest_path = RUN_DIR / "challenge2011_verified_manifest.csv"
with manifest_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(audit_rows[0]))
    writer.writeheader()
    writer.writerows(audit_rows)

groups = Counter(r["waveform_group"] for r in audit_rows)
group_labels = {}
for row in audit_rows:
    if row["target_unacceptable"] != "":
        group_labels.setdefault(row["waveform_group"], set()).add(
            row["target_unacceptable"]
        )

summary = {
    "records_verified": len(audit_rows),
    "quality_counts": dict(Counter(r["quality"] for r in audit_rows)),
    "all_digital_checksums_pass": True,
    "records_with_nonfinite_samples": sum(
        r["nonfinite_samples"] > 0 for r in audit_rows
    ),
    "records_with_constant_channels": sum(
        r["constant_channels"] > 0 for r in audit_rows
    ),
    "records_with_all_12_channels_constant": sum(
        r["constant_channels"] == 12 for r in audit_rows
    ),
    "duplicate_waveform_groups": sum(n > 1 for n in groups.values()),
    "groups_with_conflicting_binary_labels": sum(
        len(values) > 1 for values in group_labels.values()
    ),
    "source_url": URL,
    "download_sha256": sha256_file(TAR_PATH),
    "note": "Recorded SHA-256 is provenance, not publisher-hash verification.",
}
(RUN_DIR / "challenge2011_full_audit.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
print("\nAUDIT SUMMARY")
print(json.dumps(summary, indent=2))
print("\nManifest:", manifest_path)

In [ ]:
from pathlib import Path

cache = Path("/content/sentinelward-cache")

for name in (
    "challenge2011-set-a.tar.partial",
    "challenge2011-set-a.tar.gz",
):
    path = cache / name
    print(
        name,
        f"{path.stat().st_size / 1024**2:.2f} MiB"
        if path.exists() else "not present"
    )

folder = cache / "challenge2011-set-a-wfdb"
print("Staged WFDB files:",
      sum(1 for p in folder.iterdir() if p.is_file())
      if folder.exists() else 0)

In [ ]:
# RECOVERY — try the official S3 mirror with visible progress and ETA.
from pathlib import Path
from urllib.request import Request, urlopen
import gzip, json, subprocess, time

cache = Path("/content/sentinelward-cache")
cache.mkdir(parents=True, exist_ok=True)
complete = cache / "challenge2011-set-a.tar.gz"
partial = cache / "challenge2011-set-a-s3.partial"
metadata = cache / "challenge2011-set-a-s3.json"
log = cache / "challenge2011-s3-download.log"

url = (
    "https://physionet-open.s3.amazonaws.com/"
    "challenge-2011/1.0.0/set-a.tar.gz"
)

if complete.exists():
    print("DOWNLOAD READY — rerun Cell 9.", flush=True)
else:
    print("Checking official S3 mirror; timeout: 20 seconds...", flush=True)
    try:
        with urlopen(Request(url, method="HEAD"), timeout=20) as response:
            total = int(response.headers["Content-Length"])
            etag = response.headers.get("ETag")
    except Exception as error:
        raise RuntimeError(
            f"S3 availability check failed: {error}. "
            "Original partial download is unchanged; send this error."
        ) from error

    assert total > 0 and etag, "Missing remote size or version identifier."
    remote = {"url": url, "bytes": total, "etag": etag}

    # Resume only a partial file associated with this same remote object.
    if partial.exists():
        assert metadata.exists(), "S3 partial has no provenance; stop for review."
        assert json.loads(metadata.read_text()) == remote, (
            "Remote object changed; stop rather than combine versions."
        )
    metadata.write_text(json.dumps(remote, indent=2))

    initial = partial.stat().st_size if partial.exists() else 0
    assert initial <= total
    print(
        f"Size: {total / 1024**2:.1f} MiB; "
        f"already cached from S3: {initial / 1024**2:.1f} MiB.\n"
        "ETA will appear once data arrives. Transfer limit: 3 minutes.",
        flush=True
    )

    if initial < total:
        start = time.monotonic()
        with log.open("w") as errors:
            process = subprocess.Popen([
                "curl", "--location", "--fail", "--silent", "--show-error",
                "--connect-timeout", "20", "--max-time", "180",
                "--speed-limit", "10240", "--speed-time", "30",
                "--header", f"If-Match: {etag}",
                "--continue-at", "-", "--output", str(partial), url,
            ], stdout=errors, stderr=errors)

            try:
                while True:
                    try:
                        process.wait(timeout=10)
                        break
                    except subprocess.TimeoutExpired:
                        elapsed = time.monotonic() - start
                        size = partial.stat().st_size if partial.exists() else 0
                        rate = max(0, size - initial) / elapsed
                        eta = (
                            f"{max(0, total-size) / rate:.0f}s"
                            if rate > 0 else "waiting for data"
                        )
                        print(
                            f"{100*size/total:.1f}% | "
                            f"{size/1024**2:.1f}/{total/1024**2:.1f} MiB | "
                            f"elapsed {elapsed:.0f}s | estimated remaining {eta}",
                            flush=True
                        )
            finally:
                if process.poll() is None:
                    process.kill()
                    process.wait()

        if process.returncode != 0:
            raise RuntimeError(
                f"S3 transfer stopped (code {process.returncode}). "
                f"Partial retained.\n{log.read_text()[-1500:]}"
            )

    assert partial.stat().st_size == total, "Downloaded size mismatch."

    print("Download complete. Checking gzip integrity; ETA ~5–30s.", flush=True)
    with gzip.open(partial, "rb") as stream:
        while stream.read(4 * 1024**2):
            pass  # Reading to EOF verifies gzip CRC and detects truncation.

    partial.replace(complete)
    print("DOWNLOAD READY — now rerun Cell 9.", flush=True)

In [ ]:
# CELL 10 — freeze a reproducible, duplicate-group-safe 2011 split.
# ETA: approximately 2–10 seconds. No downloads or model training.
from collections import defaultdict, Counter
from pathlib import Path
import csv, hashlib, io, json, random, time

started = time.monotonic()
print("Creating grouped split — ETA 2–10 seconds...", flush=True)

SEED = 20261015
RATIOS = {"train": 0.70, "validation": 0.15, "test": 0.15}
INSPECTED = {"1002867", "1002603"}

source = RUN_DIR / "challenge2011_verified_manifest.csv"
source_bytes = source.read_bytes()
source_hash = hashlib.sha256(source_bytes).hexdigest()
rows = list(csv.DictReader(io.StringIO(source_bytes.decode("utf-8-sig"))))

assert len(rows) == 1000
assert len({r["record_id"] for r in rows}) == 1000
assert all(r["digital_checksum_pass"].lower() == "true" for r in rows)

labelled = [r for r in rows if r["target_unacceptable"] in {"0", "1"}]
excluded = [r for r in rows if r["target_unacceptable"] == ""]
assert len(labelled) == 998 and len(excluded) == 2
assert INSPECTED <= {r["record_id"] for r in labelled}

groups = defaultdict(list)
for row in labelled:
    assert row["waveform_group"], "Missing duplicate-group identifier."
    groups[row["waveform_group"]].append(row)

group_label = {}
for group, members in groups.items():
    labels = {r["target_unacceptable"] for r in members}
    assert len(labels) == 1, "Conflicting labels within a duplicate group."
    group_label[group] = next(iter(labels))

# Keep inspected examples and every exact duplicate in training.
forced_train = {
    group for group, members in groups.items()
    if any(r["record_id"] in INSPECTED for r in members)
}
assignment = {group: "train" for group in forced_train}
classes = ("0", "1")
totals = Counter(r["target_unacceptable"] for r in labelled)
counts = {split: Counter() for split in RATIOS}

for group in forced_train:
    counts["train"][group_label[group]] += len(groups[group])

targets = {
    split: {label: RATIOS[split] * totals[label] for label in classes}
    for split in RATIOS
}

# Assign whole groups, balancing record counts within each class.
# Largest groups first; seeded tie order makes the procedure repeatable.
pending = sorted(set(groups) - forced_train)
random.Random(SEED).shuffle(pending)
pending.sort(key=lambda group: -len(groups[group]))

for group in pending:
    label, size = group_label[group], len(groups[group])

    def imbalance_after(candidate):
        return sum(
            (
                counts[split][category]
                + (size if split == candidate and category == label else 0)
                - targets[split][category]
            ) ** 2 / max(targets[split][category], 1)
            for split in RATIOS for category in classes
        )

    chosen = min(RATIOS, key=imbalance_after)
    assignment[group] = chosen
    counts[chosen][label] += size

output_rows = []
for row in sorted(rows, key=lambda r: r["record_id"]):
    split = (
        assignment[row["waveform_group"]]
        if row["target_unacceptable"] in classes else "excluded_unlabelled"
    )
    output_rows.append({
        "record_id": row["record_id"],
        "target_unacceptable": row["target_unacceptable"],
        "waveform_group": row["waveform_group"],
        "split": split,
    })

# Assert separation, class presence and inspected-example placement.
group_splits = defaultdict(set)
for row in output_rows:
    if row["split"] in RATIOS:
        group_splits[row["waveform_group"]].add(row["split"])
assert all(len(parts) == 1 for parts in group_splits.values())
assert all(counts[s][label] > 0 for s in RATIOS for label in classes)
assert all(
    row["split"] == "train"
    for row in output_rows if row["record_id"] in INSPECTED
)

summary = {}
for split in RATIOS:
    members = [
        r for r in labelled
        if assignment[r["waveform_group"]] == split
    ]
    summary[split] = {
        "records": len(members),
        "acceptable": counts[split]["0"],
        "unacceptable": counts[split]["1"],
        "waveform_groups": len({r["waveform_group"] for r in members}),
        "records_with_all_channels_constant": sum(
            int(r["constant_channels"]) == 12 for r in members
        ),
    }

buffer = io.StringIO(newline="")
writer = csv.DictWriter(buffer, fieldnames=list(output_rows[0]))
writer.writeheader()
writer.writerows(output_rows)
split_bytes = buffer.getvalue().encode("utf-8")
split_hash = hashlib.sha256(split_bytes).hexdigest()

provenance = {
    "seed": SEED,
    "target_ratios": RATIOS,
    "algorithm": "largest-group-first class-count balancing, version 1",
    "source_manifest_sha256": source_hash,
    "split_manifest_sha256": split_hash,
    "inspected_records_forced_to_train": sorted(INSPECTED),
    "excluded_records": sorted(r["record_id"] for r in excluded),
    "summary": summary,
    "limitation": "Record/known-duplicate separation; patient identity unverified.",
}

split_path = RUN_DIR / "challenge2011_frozen_split.csv"
provenance_path = RUN_DIR / "challenge2011_frozen_split_provenance.json"
provenance_bytes = json.dumps(provenance, indent=2).encode("utf-8")

# Never silently replace an existing split with a different one.
for path, content in (
    (split_path, split_bytes),
    (provenance_path, provenance_bytes),
):
    if path.exists():
        assert path.read_bytes() == content, (
            f"Existing frozen file differs: {path.name}. Stop for review."
        )

split_path.write_bytes(split_bytes)
provenance_path.write_bytes(provenance_bytes)

print("\nSPLIT SUMMARY")
print(json.dumps(summary, indent=2))
print("Excluded unlabelled:", provenance["excluded_records"])
print("Duplicate-group overlap: 0")
print("Previously inspected examples: training only")
print("Split SHA-256:", split_hash)
print("Saved:", split_path)
print(f"Completed in {time.monotonic() - started:.1f} seconds.")

In [ ]:
# CELL 11 — deterministic ECG-quality features; TRAIN + VALIDATION ONLY.
# Initial ETA: ~30–120 seconds. Progress updates every 100 records.
from pathlib import Path
import hashlib, json, time
import numpy as np
import pandas as pd
from scipy.signal import welch
import wfdb

started = time.monotonic()
print("Extracting development features — initial ETA ~30–120s.", flush=True)

FEATURE_VERSION = "ecg12-quality-v1"
WFDB_ROOT = Path("/content/sentinelward-cache/challenge2011-set-a-wfdb")
split_path = RUN_DIR / "challenge2011_frozen_split.csv"
provenance_path = RUN_DIR / "challenge2011_frozen_split_provenance.json"

assert WFDB_ROOT.is_dir(), "Local waveforms missing; do not recreate the split."
provenance = json.loads(provenance_path.read_text())
split_hash = hashlib.sha256(split_path.read_bytes()).hexdigest()
assert split_hash == provenance["split_manifest_sha256"], "Split file changed."

splits = pd.read_csv(
    split_path, dtype={"record_id": str, "waveform_group": str}
)
development = splits[splits["split"].isin(["train", "validation"])].copy()
assert len(development) == 848
assert development["record_id"].is_unique

def extract_quality_features(x, fs):
    """Summarise per-lead measurements into whole-record features."""
    assert x.shape == (5000, 12)
    assert np.isfinite(x).all(), "Unexpected missing values; stop for review."

    difference = np.diff(x, axis=0)
    constant = np.ptp(x, axis=0) == 0

    # Original physical amplitudes: do not erase flat/noisy-signal evidence.
    lead_features = {
        "std_mv": np.std(x, axis=0),
        "range_mv": np.ptp(x, axis=0),
        "iqr_mv": np.percentile(x, 75, axis=0)
                  - np.percentile(x, 25, axis=0),
        "abs_mean_mv": np.abs(np.mean(x, axis=0)),
        "rms_mv": np.sqrt(np.mean(x ** 2, axis=0)),
        "median_abs_step_mv": np.median(np.abs(difference), axis=0),
        "p95_abs_step_mv": np.percentile(np.abs(difference), 95, axis=0),
        "unchanged_step_fraction": np.mean(difference == 0, axis=0),
    }

    # Spectral descriptors, not yet clinically validated SQI thresholds.
    frequencies, power = welch(
        x, fs=fs, nperseg=1000, noverlap=500,
        detrend="constant", axis=0
    )

    def band(low, high):
        return power[(frequencies >= low) & (frequencies < high)].sum(axis=0)

    def ratio(numerator, denominator):
        # Zero-power leads remain identifiable through constant_leads.
        return np.divide(
            numerator, denominator,
            out=np.zeros_like(numerator), where=denominator > 0
        )

    total_power = power.sum(axis=0)
    lead_features.update({
        "power_fraction_below_0p5hz": ratio(band(0, 0.5), total_power),
        "power_fraction_40_100hz": ratio(band(40, 100), total_power),
        "power_ratio_5_15_to_5_40hz": ratio(band(5, 15), band(5, 40)),
    })

    result = {
        "constant_leads": int(constant.sum()),
        "all_leads_constant": int(constant.all()),
    }
    for name, values in lead_features.items():
        result[f"{name}_min"] = float(np.min(values))
        result[f"{name}_median"] = float(np.median(values))
        result[f"{name}_max"] = float(np.max(values))
    return result

feature_rows, source_hashes = [], {}
for index, row in enumerate(development.itertuples(index=False), start=1):
    base = WFDB_ROOT / row.record_id
    record = wfdb.rdrecord(str(base), physical=True)

    assert record.fs == 500 and record.n_sig == 12
    assert all(unit == "mV" for unit in record.units)
    assert set(record.file_name) == {row.record_id + ".dat"}

    feature_rows.append({
        "record_id": row.record_id,
        "waveform_group": row.waveform_group,
        "split": row.split,
        "target_unacceptable": int(row.target_unacceptable),
        **extract_quality_features(record.p_signal, record.fs),
    })

    # Record exactly which local source bytes produced these features.
    for extension in (".hea", ".dat"):
        path = base.with_suffix(extension)
        source_hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()

    if index % 100 == 0 or index == len(development):
        elapsed = time.monotonic() - started
        remaining = elapsed / index * (len(development) - index)
        print(
            f"{index}/{len(development)} | elapsed {elapsed:.1f}s | "
            f"estimated remaining {remaining:.1f}s",
            flush=True
        )

features = pd.DataFrame(feature_rows)
metadata_columns = [
    "record_id", "waveform_group", "split", "target_unacceptable"
]
feature_columns = [c for c in features if c not in metadata_columns]

assert np.isfinite(features[feature_columns].to_numpy()).all()
assert not (features["split"] == "test").any()
assert set(features["record_id"]) == set(development["record_id"])

output = RUN_DIR / f"challenge2011_development_features_{FEATURE_VERSION}.csv"
csv_bytes = features.to_csv(index=False).encode("utf-8")

# Prevent silently replacing an earlier result with different values.
if output.exists():
    assert output.read_bytes() == csv_bytes, (
        "Existing features differ. Stop and review the environment/version."
    )
output.write_bytes(csv_bytes)

feature_metadata = {
    "feature_version": FEATURE_VERSION,
    "split_sha256": split_hash,
    "feature_csv_sha256": hashlib.sha256(csv_bytes).hexdigest(),
    "feature_columns": feature_columns,
    "source_file_sha256": source_hashes,
    "scope": "Whole-record 12-lead ECG quality; train and validation only.",
}
(RUN_DIR / f"{FEATURE_VERSION}_metadata.json").write_text(
    json.dumps(feature_metadata, indent=2), encoding="utf-8"
)

print("\nFEATURE SUMMARY")
print(features.groupby("split").agg(
    records=("record_id", "size"),
    unacceptable=("target_unacceptable", "sum"),
    fully_constant=("all_leads_constant", "sum"),
).to_string())
print("Numeric feature columns:", len(feature_columns))
print("Nonfinite feature values: 0")
print("Test records processed: 0")
print("Saved:", output)

In [ ]:
# CELL 12 — fit a fixed baseline and evaluate VALIDATION ONLY.
# ETA: ~5–30 seconds. No threshold search or test-set evaluation.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.metadata, json, time
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    balanced_accuracy_score, roc_auc_score, average_precision_score
)

started = time.monotonic()
print("Checking inputs and fitting baseline — ETA ~5–30s.", flush=True)

VERSION = "ecg12-quality-v1"
feature_path = RUN_DIR / f"challenge2011_development_features_{VERSION}.csv"
meta = json.loads((RUN_DIR / f"{VERSION}_metadata.json").read_text())
split_path = RUN_DIR / "challenge2011_frozen_split.csv"

assert hashlib.sha256(feature_path.read_bytes()).hexdigest() == (
    meta["feature_csv_sha256"]
), "Feature file changed."
assert hashlib.sha256(split_path.read_bytes()).hexdigest() == (
    meta["split_sha256"]
), "Frozen split changed."

data = pd.read_csv(
    feature_path, dtype={"record_id": str, "waveform_group": str}
)
columns = meta["feature_columns"]
assert len(columns) == 35
assert not set(columns) & {
    "record_id", "waveform_group", "split", "target_unacceptable"
}
assert set(data["split"]) == {"train", "validation"}
assert np.isfinite(data[columns].to_numpy()).all()

train = data[data["split"] == "train"].copy()
valid = data[data["split"] == "validation"].copy()
assert len(train) == 698 and len(valid) == 150
assert not set(train["waveform_group"]) & set(valid["waveform_group"])

# Verify identical-group members have identical features and targets.
agreement = data.groupby("waveform_group")[
    columns + ["target_unacceptable"]
].nunique(dropna=False)
assert (agreement <= 1).all().all(), (
    "A duplicate group has inconsistent features or labels."
)

train_unique = (
    train.sort_values("record_id")
    .drop_duplicates("waveform_group")
)
assert len(train_unique) == 649
assert set(train_unique["target_unacceptable"]) == {0, 1}

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=3,
    max_features="sqrt",
    class_weight="balanced",
    random_state=20261015,
    n_jobs=2,
)

print(
    f"Training on {len(train_unique)} unique waveform groups...",
    flush=True
)
model.fit(train_unique[columns], train_unique["target_unacceptable"])

positive_column = list(model.classes_).index(1)
valid["rf_score"] = model.predict_proba(valid[columns])[:, positive_column]
valid["rf_prediction"] = (valid["rf_score"] >= 0.50).astype(int)
print(f"Model fitted; elapsed {time.monotonic()-started:.1f}s.", flush=True)

def evaluate(frame, scores, model_name, scope):
    y = frame["target_unacceptable"].to_numpy(dtype=int)
    scores = np.asarray(scores, dtype=float)
    prediction = (scores >= 0.50).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        y, prediction, labels=[0, 1]
    ).ravel()
    both_classes = len(np.unique(y)) == 2
    return {
        "model": model_name,
        "scope": scope,
        "n": len(y),
        "unacceptable": int(y.sum()),
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "recall_unacceptable": float(recall_score(
            y, prediction, zero_division=0
        )),
        "specificity_acceptable": (
            float(tn / (tn + fp)) if tn + fp else None
        ),
        "precision_unacceptable": float(precision_score(
            y, prediction, zero_division=0
        )),
        "f1_unacceptable": float(f1_score(
            y, prediction, zero_division=0
        )),
        "balanced_accuracy": (
            float(balanced_accuracy_score(y, prediction))
            if both_classes else None
        ),
        "auroc": float(roc_auc_score(y, scores)) if both_classes else None,
        "average_precision": (
            float(average_precision_score(y, scores))
            if both_classes else None
        ),
    }

unique_valid = (
    valid.sort_values("record_id").drop_duplicates("waveform_group")
)
scopes = {
    "all_records": valid,
    "nonconstant_records": valid[valid["all_leads_constant"] == 0],
    "unique_groups": unique_valid,
    "nonconstant_unique_groups": unique_valid[
        unique_valid["all_leads_constant"] == 0
    ],
}

results = []
for scope, frame in scopes.items():
    if frame.empty:
        continue
    results.append(evaluate(frame, frame["rf_score"], "random_forest", scope))
    results.append(evaluate(
        frame, np.zeros(len(frame)), "always_acceptable", scope
    ))

# Each execution gets its own output directory.
output_dir = RUN_DIR / (
    "m1_baseline_" +
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
output_dir.mkdir(parents=True, exist_ok=False)

metrics = pd.DataFrame(results)
metrics.to_csv(output_dir / "validation_metrics.csv", index=False)
valid[
    ["record_id", "waveform_group", "target_unacceptable",
     "all_leads_constant", "rf_score", "rf_prediction"]
].to_csv(output_dir / "validation_predictions.csv", index=False)

joblib.dump({
    "model": model,
    "feature_columns": columns,
    "feature_version": VERSION,
    "threshold": 0.50,
    "target": "whole-record 12-lead ECG unacceptable quality",
    "calibrated": False,
}, output_dir / "random_forest_baseline.joblib")

configuration = {
    "model_parameters": model.get_params(),
    "training_records_before_deduplication": len(train),
    "training_groups_used": len(train_unique),
    "split_sha256": meta["split_sha256"],
    "feature_csv_sha256": meta["feature_csv_sha256"],
    "threshold": 0.50,
    "test_used": False,
    "packages": {
        name: importlib.metadata.version(name)
        for name in ("scikit-learn", "numpy", "pandas", "scipy", "joblib")
    },
}
(output_dir / "configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)

print("\nVALIDATION RESULTS — threshold 0.50")
print(metrics.round(4).to_string(index=False))
print("\nFN = unacceptable ECG classified as acceptable.")
print("FP = acceptable ECG classified as unacceptable.")
print("Average precision uses ranking scores; it is not trapezoidal PR-AUC.")
print("Test set remains unused. Model scores are uncalibrated.")
print("Saved:", output_dir)
print(f"Completed in {time.monotonic()-started:.1f} seconds.")

In [ ]:
# CELL 13 — five-seed stability check, VALIDATION ONLY.
# ETA: ~5–20 seconds. No parameter or threshold search.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, time
import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone

required = [
    "model", "train_unique", "valid", "columns",
    "evaluate", "RUN_DIR", "meta"
]
assert all(name in globals() for name in required), (
    "Run Cell 12 first in this runtime."
)

started = time.monotonic()
SEEDS = [20261015, 20261016, 20261017, 20261018, 20261019]
THRESHOLD = 0.50

print("Five-seed validation check — ETA ~5–20s.", flush=True)
assert model.random_state == SEEDS[0]
assert len(train_unique) == 649 and len(valid) == 150
assert set(valid["split"]) == {"validation"}
assert not set(train_unique["waveform_group"]) & set(valid["waveform_group"])

feature_path = RUN_DIR / (
    "challenge2011_development_features_ecg12-quality-v1.csv"
)
assert hashlib.sha256(feature_path.read_bytes()).hexdigest() == (
    meta["feature_csv_sha256"]
)
assert hashlib.sha256(
    (RUN_DIR / "challenge2011_frozen_split.csv").read_bytes()
).hexdigest() == meta["split_sha256"]

destination = RUN_DIR / (
    "m1_five_seed_" +
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
)
destination.mkdir(parents=True, exist_ok=False)

models, scores, results = [], [], []
validation = valid.reset_index(drop=True).copy()

unique_mask = ~validation["waveform_group"].duplicated()
nonconstant_mask = validation["all_leads_constant"].eq(0)
masks = {
    "all_records": np.ones(len(validation), dtype=bool),
    "nonconstant_records": nonconstant_mask.to_numpy(),
    "unique_groups": unique_mask.to_numpy(),
    "nonconstant_unique_groups": (
        unique_mask & nonconstant_mask
    ).to_numpy(),
}

for index, seed in enumerate(SEEDS, start=1):
    fitted = model if index == 1 else clone(model).set_params(random_state=seed)

    if index > 1:
        fitted.fit(
            train_unique[columns],
            train_unique["target_unacceptable"]
        )

    positive_column = list(fitted.classes_).index(1)
    score = fitted.predict_proba(validation[columns])[:, positive_column]
    models.append(fitted)
    scores.append(score)

    for scope, mask in masks.items():
        result = evaluate(
            validation.loc[mask], score[mask], "random_forest", scope
        )
        result["seed"] = seed
        results.append(result)

    elapsed = time.monotonic() - started
    remaining = elapsed / index * (len(SEEDS) - index)
    print(
        f"Seed {index}/5 complete | elapsed {elapsed:.1f}s | "
        f"estimated remaining {remaining:.1f}s",
        flush=True
    )

metrics = pd.DataFrame(results)
metrics.to_csv(destination / "validation_metrics_by_seed.csv", index=False)

# Save every model, without selecting the seed with the highest score.
joblib.dump({
    "models": models,
    "seeds": SEEDS,
    "feature_columns": columns,
    "feature_version": "ecg12-quality-v1",
    "threshold": THRESHOLD,
    "calibrated": False,
    "scope": "whole-record 12-lead ECG quality",
}, destination / "five_seed_models.joblib")

prediction_table = validation[
    ["record_id", "waveform_group", "target_unacceptable",
     "all_leads_constant"]
].copy()
for seed, score in zip(SEEDS, scores):
    prediction_table[f"score_seed_{seed}"] = score
prediction_table.to_csv(
    destination / "validation_predictions_by_seed.csv", index=False
)

configuration = {
    "seeds": SEEDS,
    "threshold": THRESHOLD,
    "model_parameters_except_seed": {
        k: v for k, v in model.get_params().items() if k != "random_state"
    },
    "split_sha256": meta["split_sha256"],
    "feature_csv_sha256": meta["feature_csv_sha256"],
    "test_used": False,
    "selection_policy": "Report all five seeds; do not select the best seed.",
}
(destination / "configuration.json").write_text(
    json.dumps(configuration, indent=2), encoding="utf-8"
)

reported = [
    "recall_unacceptable", "specificity_acceptable",
    "f1_unacceptable", "auroc", "average_precision"
]
summary = metrics.groupby("scope")[reported].agg(["mean", "std"])

print("\nFIVE-SEED VALIDATION SUMMARY — mean and sample SD")
print(summary.round(4).to_string())

print("\nMISSES AND FALSE FLAGS BY SEED — unique groups")
print(metrics.loc[
    metrics["scope"].eq("unique_groups"),
    ["seed", "FN", "FP", "TP", "TN"]
].to_string(index=False))

print("\nSD measures randomness across fits, not patient-level uncertainty.")
print("Test records processed: 0")
print("Saved:", destination)
print(f"Completed in {time.monotonic() - started:.1f}s.")

In [ ]:
# CELL 14 — frozen five-seed TEST evaluation; no fitting or tuning.
# ETA: ~5–20 seconds. Re-runs display the saved results.
from pathlib import Path
import hashlib, json, time
import joblib
import numpy as np
import pandas as pd
import wfdb

assert callable(globals().get("extract_quality_features")), "Run Cell 11 first."
assert callable(globals().get("evaluate")), "Run Cell 12 first."

BASELINE_DIR = RUN_DIR / "m1_five_seed_20261015T143443_468300Z"
MODEL_FILE = BASELINE_DIR / "five_seed_models.joblib"
TEST_DIR = BASELINE_DIR / "frozen_test"
WFDB_ROOT = Path("/content/sentinelward-cache/challenge2011-set-a-wfdb")

def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def display_test_results(results):
    measures = [
        "recall_unacceptable", "specificity_acceptable",
        "f1_unacceptable", "auroc", "average_precision"
    ]
    print("\nTEST RESULTS — mean and sample SD across all five seeds")
    print(results.groupby("scope")[measures].agg(
        ["mean", "std"]
    ).round(4).to_string())

    print("\nTEST CONFUSION COUNTS — unique groups")
    print(results.loc[
        results["scope"].eq("unique_groups"),
        ["seed", "n", "unacceptable", "TN", "FP", "FN", "TP"]
    ].to_string(index=False))

def run_frozen_test():
    started = time.monotonic()
    print("Frozen test evaluation — ETA ~5–20s.", flush=True)

    config = json.loads((BASELINE_DIR / "configuration.json").read_text())
    split_path = RUN_DIR / "challenge2011_frozen_split.csv"
    assert file_sha256(split_path) == config["split_sha256"]

    feature_path = RUN_DIR / (
        "challenge2011_development_features_ecg12-quality-v1.csv"
    )
    assert file_sha256(feature_path) == config["feature_csv_sha256"]

    policy = {
        "model_sha256": file_sha256(MODEL_FILE),
        "split_sha256": config["split_sha256"],
        "development_features_sha256": config["feature_csv_sha256"],
        "seeds": config["seeds"],
        "threshold": 0.50,
        "evaluation": "All five fixed models, separately; no best-seed selection.",
    }
    TEST_DIR.mkdir(parents=True, exist_ok=True)
    policy_path = TEST_DIR / "frozen_policy.json"

    if policy_path.exists():
        assert json.loads(policy_path.read_text()) == policy, (
            "Frozen inputs changed. Stop rather than overwrite the test."
        )
    else:
        # Record the evaluation policy before calculating any test predictions.
        policy_path.write_text(json.dumps(policy, indent=2), encoding="utf-8")

    metrics_path = TEST_DIR / "test_metrics_by_seed.csv"
    if metrics_path.exists():
        display_test_results(pd.read_csv(metrics_path))
        print("\nDisplaying saved evaluation; nothing was refitted or rescored.")
        return

    bundle = joblib.load(MODEL_FILE)  # Our own saved artifact.
    feature_names = bundle["feature_columns"]
    assert bundle["threshold"] == 0.50
    assert bundle["seeds"] == config["seeds"]
    assert len(bundle["models"]) == 5

    # Check the current feature function against saved development examples.
    development = pd.read_csv(feature_path, dtype={"record_id": str})
    for record_id in ("1002867", "1002603"):
        reference = development.loc[
            development["record_id"].eq(record_id), feature_names
        ]
        assert len(reference) == 1
        record = wfdb.rdrecord(str(WFDB_ROOT / record_id), physical=True)
        current = extract_quality_features(record.p_signal, record.fs)
        np.testing.assert_allclose(
            [current[name] for name in feature_names],
            reference.iloc[0].to_numpy(dtype=float),
            rtol=1e-10, atol=1e-12,
        )

    splits = pd.read_csv(
        split_path, dtype={"record_id": str, "waveform_group": str}
    )
    test = splits[splits["split"].eq("test")].sort_values("record_id")
    assert len(test) == 150
    assert not set(test["waveform_group"]) & set(
        splits.loc[splits["split"].isin(["train", "validation"]), "waveform_group"]
    )

    feature_rows = []
    for number, row in enumerate(test.itertuples(index=False), start=1):
        base = WFDB_ROOT / row.record_id
        header = wfdb.rdheader(str(base))
        assert header.fs == 500 and header.n_sig == 12
        assert header.sig_len == 5000
        assert set(header.file_name) == {row.record_id + ".dat"}
        assert all(unit == "mV" for unit in header.units)

        # Match the waveform signature used when creating the frozen groups.
        signature = {
            "dat_sha256": file_sha256(base.with_suffix(".dat")),
            "fs": header.fs,
            "samples": header.sig_len,
            "leads": header.sig_name,
            "units": header.units,
            "format": header.fmt,
            "gain": header.adc_gain,
            "baseline": header.baseline,
        }
        group_hash = hashlib.sha256(
            json.dumps(signature, sort_keys=True).encode()
        ).hexdigest()
        assert group_hash == row.waveform_group, (
            f"Source changed for {row.record_id}."
        )

        digital = wfdb.rdrecord(str(base), physical=False).d_signal
        np.testing.assert_array_equal(
            digital.astype(np.int64).sum(axis=0) & 0xFFFF,
            np.asarray(header.checksum, dtype=np.int64) & 0xFFFF,
        )
        record = wfdb.rdrecord(str(base), physical=True)
        values = extract_quality_features(record.p_signal, record.fs)
        assert set(values) == set(feature_names)

        feature_rows.append({
            "record_id": row.record_id,
            "waveform_group": row.waveform_group,
            "target_unacceptable": int(row.target_unacceptable),
            **values,
        })
        if number % 50 == 0:
            elapsed = time.monotonic() - started
            print(
                f"Features {number}/150 | elapsed {elapsed:.1f}s | "
                f"estimated feature-stage remaining "
                f"{elapsed/number*(150-number):.1f}s",
                flush=True
            )

    features = pd.DataFrame(feature_rows)
    assert np.isfinite(features[feature_names].to_numpy()).all()
    assert features["target_unacceptable"].sum() == 34

    unique = ~features["waveform_group"].duplicated()
    nonconstant = features["all_leads_constant"].eq(0)
    masks = {
        "all_records": np.ones(len(features), dtype=bool),
        "nonconstant_records": nonconstant.to_numpy(),
        "unique_groups": unique.to_numpy(),
        "nonconstant_unique_groups": (unique & nonconstant).to_numpy(),
    }

    predictions = features[
        ["record_id", "waveform_group", "target_unacceptable",
         "all_leads_constant"]
    ].copy()
    results = []

    for index, (seed, model) in enumerate(
        zip(bundle["seeds"], bundle["models"]), start=1
    ):
        positive = list(model.classes_).index(1)
        scores = model.predict_proba(features[feature_names])[:, positive]
        predictions[f"score_seed_{seed}"] = scores

        for scope, mask in masks.items():
            result = evaluate(
                features.loc[mask], scores[mask], "random_forest", scope
            )
            result["seed"] = seed
            results.append(result)
        print(f"Scored fixed model {index}/5.", flush=True)

    results = pd.DataFrame(results)
    features.to_csv(TEST_DIR / "test_features.csv", index=False)
    predictions.to_csv(TEST_DIR / "test_predictions.csv", index=False)

    # Write final metrics last, so their presence marks completed evaluation.
    temporary = TEST_DIR / "test_metrics_by_seed.partial.csv"
    results.to_csv(temporary, index=False)
    temporary.replace(metrics_path)

    display_test_results(results)
    print("\nNo model fitting or threshold selection used test data.")
    print("Seed SD is not a confidence interval for population performance.")
    print("Saved:", TEST_DIR)
    print(f"Completed in {time.monotonic()-started:.1f}s.")

run_frozen_test()

In [ ]:
# CELL 15 — VTaC training examples: publisher hashes + pre-alarm signals.
# Initial ETA ~10–90s; Drive latency can vary. No GPU or training.
from pathlib import Path, PurePosixPath
import hashlib, io, json, time, zipfile
import numpy as np
import pandas as pd
import wfdb

started = time.monotonic()
print("VTaC source verification — initial ETA ~10–90s.", flush=True)
assert "ARCHIVES" in globals() and "RUN_DIR" in globals()

LOCAL = Path("/content/sentinelward-cache/vtac-training-examples")
LOCAL.mkdir(parents=True, exist_ok=True)
report = []

with zipfile.ZipFile(ARCHIVES["vtac-1.1"]) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), "Duplicate archive paths."

    def unique_member(basename):
        matches = [
            name for name in names
            if PurePosixPath(name).name == basename
        ]
        assert len(matches) == 1, f"Ambiguous/missing {basename}: {matches}"
        return matches[0]

    checksum_member = unique_member("SHA256SUMS.txt")
    root = str(PurePosixPath(checksum_member).parent)

    checksums = {}
    for line in archive.read(checksum_member).decode("utf-8-sig").splitlines():
        if not line.strip():
            continue
        digest, filename = line.split(maxsplit=1)
        filename = filename.lstrip("*")
        if filename.startswith("./"):
            filename = filename[2:]
        assert len(digest) == 64
        assert all(c in "0123456789abcdefABCDEF" for c in digest)
        assert filename not in checksums, "Duplicate checksum entry."
        checksums[filename] = digest.lower()

    def verified_bytes(member):
        relative = str(PurePosixPath(member).relative_to(root))
        candidates = [key for key in (relative, member) if key in checksums]
        assert candidates, f"No publisher SHA-256 entry for {member}"
        expected = {checksums[key] for key in candidates}
        assert len(expected) == 1, "Conflicting publisher hashes."
        payload = archive.read(member)
        actual = hashlib.sha256(payload).hexdigest()
        assert actual == next(iter(expected)), f"SHA-256 mismatch: {member}"
        return payload, actual

    labels_bytes, _ = verified_bytes(unique_member("event_labels.csv"))
    splits_bytes, _ = verified_bytes(unique_member("benchmark_data_split.csv"))
    labels = pd.read_csv(
        io.BytesIO(labels_bytes), dtype=str, keep_default_na=False
    )
    splits = pd.read_csv(
        io.BytesIO(splits_bytes), dtype=str, keep_default_na=False
    )
    assert set(labels["decision"]) == {"False", "True"}

    joined = labels.merge(
        splits, on=["record", "event"], how="outer",
        validate="one_to_one", indicator=True
    )
    assert len(joined) == 5037 and joined["_merge"].eq("both").all()
    training = joined[joined["split"].str.lower().eq("train")]
    assert len(training) == 4060

    examples = [
        training[training["decision"].eq(label)]
        .sort_values(["record", "event"]).iloc[0]
        for label in ("False", "True")
    ]
    print("Official labels and splits: publisher SHA-256 PASS.", flush=True)

    for index, row in enumerate(examples, start=1):
        record_id, event_id = row["record"], row["event"]
        assert Path(record_id).name == record_id
        assert Path(event_id).name == event_id

        header_member = (
            f"{root}/annotated_waveforms/{record_id}/{event_id}.hea"
        )
        assert header_member in names, f"Missing header: {header_member}"

        folder = LOCAL / record_id
        folder.mkdir(parents=True, exist_ok=True)
        header_bytes, header_hash = verified_bytes(header_member)
        (folder / f"{event_id}.hea").write_bytes(header_bytes)
        header = wfdb.rdheader(str(folder / event_id))

        assert header.record_name == event_id
        assert header.fs == 250 and header.sig_len == 90000
        source_hashes = {header_member: header_hash}

        for filename in sorted(set(header.file_name)):
            assert Path(filename).name == filename
            assert "/" not in filename and "\\" not in filename
            member = str(PurePosixPath(header_member).parent / filename)
            payload, digest = verified_bytes(member)
            (folder / filename).write_bytes(payload)
            source_hashes[member] = digest

        # VTaC event protocol: alarm at 300s; real-time input [290s, 300s).
        # The protocol supplies alarm timing; header duration alone does not.
        start_sample = int(290 * header.fs)
        stop_sample = int(300 * header.fs)
        waveform = wfdb.rdrecord(
            str(folder / event_id),
            sampfrom=start_sample, sampto=stop_sample,
            physical=True
        )
        x = waveform.p_signal
        assert x.shape == (2500, header.n_sig)

        channels = []
        for channel, name in enumerate(waveform.sig_name):
            values = x[:, channel]
            finite = values[np.isfinite(values)]
            channels.append({
                "name": name,
                "unit": waveform.units[channel],
                "nonfinite_samples": int((~np.isfinite(values)).sum()),
                "constant_finite_values": (
                    bool(np.ptp(finite) == 0) if finite.size else None
                ),
                "min": float(finite.min()) if finite.size else None,
                "max": float(finite.max()) if finite.size else None,
            })

        result = {
            "record_id": record_id,
            "event_id": event_id,
            "split": "train",
            "true_vt_alarm": row["decision"] == "True",
            "publisher_hashes_pass": True,
            "source_sha256": source_hashes,
            "input_interval_seconds": [290, 300],
            "native_shape": list(x.shape),
            "sampling_rate_hz": float(waveform.fs),
            "channels": channels,
        }
        report.append(result)
        print(f"\nExample {index}/2:", flush=True)
        print(json.dumps(
            {k: v for k, v in result.items() if k != "source_sha256"},
            indent=2
        ))
        elapsed = time.monotonic() - started
        print(f"Elapsed: {elapsed:.1f}s", flush=True)

output = RUN_DIR / "vtac_training_examples_verified.json"
output.write_text(
    json.dumps(report, indent=2, allow_nan=False), encoding="utf-8"
)
print("\nSaved:", output)
print("VTaC test waveforms accessed: 0")
print("Full VTaC archive verification remains outstanding.")
print(f"Completed in {time.monotonic()-started:.1f}s.")

In [ ]:
# CELL 16 — verified VTaC development-header manifest.
# Initial ETA: ~1–5 minutes. Headers only; no waveform downloads or training.
from pathlib import Path, PurePosixPath
from collections import Counter
import hashlib, io, json, time, zipfile
import pandas as pd
import wfdb

started = time.monotonic()
print("VTaC header inventory — initial ETA ~1–5 minutes.", flush=True)
assert "ARCHIVES" in globals() and "RUN_DIR" in globals()

HEADER_CACHE = Path("/content/sentinelward-cache/vtac-headers")
HEADER_CACHE.mkdir(parents=True, exist_ok=True)
manifest_rows = []

# Proposed deterministic ECG preference; eligibility remains provisional
# until actual waveform missingness is checked.
ECG_PREFERENCE = ["II", "V", "I", "III", "aVR", "aVL", "aVF",
                  "V1", "V2", "V3", "V4", "V5", "V6"]

with zipfile.ZipFile(ARCHIVES["vtac-1.1"]) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)), "Duplicate archive paths."
    name_set = set(names)

    def unique_name(basename):
        matches = [n for n in names if PurePosixPath(n).name == basename]
        assert len(matches) == 1, f"Missing/ambiguous {basename}"
        return matches[0]

    checksum_name = unique_name("SHA256SUMS.txt")
    archive_root = str(PurePosixPath(checksum_name).parent)
    checksum_bytes = archive.read(checksum_name)
    hashes = {}

    for line in checksum_bytes.decode("utf-8-sig").splitlines():
        if not line.strip():
            continue
        digest, filename = line.split(maxsplit=1)
        filename = filename.lstrip("*")
        if filename.startswith("./"):
            filename = filename[2:]
        assert filename not in hashes, "Duplicate checksum entry."
        assert len(digest) == 64
        assert all(c in "0123456789abcdefABCDEF" for c in digest)
        hashes[filename] = digest.lower()

    def expected_hash(member):
        relative = str(PurePosixPath(member).relative_to(archive_root))
        matches = {hashes[k] for k in (relative, member) if k in hashes}
        assert len(matches) == 1, f"Missing/conflicting checksum: {member}"
        return next(iter(matches))

    def verified_read(member):
        payload = archive.read(member)
        assert hashlib.sha256(payload).hexdigest() == expected_hash(member), (
            f"Publisher checksum failed: {member}"
        )
        return payload

    labels_payload = verified_read(unique_name("event_labels.csv"))
    splits_payload = verified_read(unique_name("benchmark_data_split.csv"))
    labels = pd.read_csv(io.BytesIO(labels_payload), dtype=str)
    splits = pd.read_csv(io.BytesIO(splits_payload), dtype=str)

    joined = labels.merge(
        splits, on=["record", "event"], how="outer",
        validate="one_to_one", indicator=True
    )
    assert len(joined) == 5037 and joined["_merge"].eq("both").all()
    assert joined.groupby("record")["split"].nunique().max() == 1
    assert set(joined["decision"]) == {"True", "False"}
    print("Original split names:", joined["split"].value_counts().to_dict(),
          flush=True)

    aliases = {
        "train": "train", "training": "train",
        "val": "validation", "valid": "validation", "validation": "validation",
        "test": "test", "testing": "test",
    }
    raw_split = joined["split"].astype(str).str.strip().str.lower()
    unknown = sorted(set(raw_split) - set(aliases))
    assert not unknown, f"Unrecognised split names: {unknown}"

    joined["split"] = raw_split.map(aliases)

    expected = {"train": 4060, "validation": 495, "test": 482}
    actual = joined["split"].value_counts().to_dict()
    assert actual == expected, f"Unexpected split counts: {actual}"
    assert joined.groupby("record")["split"].nunique().max() == 1

    development = joined[
        joined["split"].isin(["train", "validation"])
    ].sort_values(["split", "record", "event"])

    print("Verified split counts:", actual, flush=True)

    for number, row in enumerate(development.itertuples(index=False), start=1):
        assert Path(row.record).name == row.record
        assert Path(row.event).name == row.event

        member = (
            f"{archive_root}/annotated_waveforms/"
            f"{row.record}/{row.event}.hea"
        )
        assert member in name_set, f"Missing header: {member}"
        digest = expected_hash(member)

        local = HEADER_CACHE / row.record / f"{row.event}.hea"
        local.parent.mkdir(parents=True, exist_ok=True)
        if not local.exists() or hashlib.sha256(
            local.read_bytes()
        ).hexdigest() != digest:
            local.write_bytes(verified_read(member))

        header = wfdb.rdheader(str(local.with_suffix("")))
        assert header.record_name == row.event
        assert header.fs == 250 and header.sig_len == 90000, (
            f"Unexpected sampling rate/duration: {row.event}"
        )
        assert len(header.sig_name) == len(set(header.sig_name)), (
            f"Duplicate channel names: {row.event}"
        )

        parent = PurePosixPath(member).parent
        signal_members = []
        for filename in sorted(set(header.file_name)):
            assert PurePosixPath(filename).name == filename
            assert "/" not in filename and "\\" not in filename
            signal_member = str(parent / filename)
            assert signal_member in name_set, f"Missing data: {signal_member}"
            expected_hash(signal_member)  # Existence only; bytes checked later.
            signal_members.append(signal_member)

        available_ecg = [
            lead for lead in ECG_PREFERENCE if lead in header.sig_name
        ]
        chosen_ecg = available_ecg[0] if available_ecg else None
        ppg_index = (
            header.sig_name.index("PLETH")
            if "PLETH" in header.sig_name else None
        )

        manifest_rows.append({
            "record_id": row.record,
            "event_id": row.event,
            "split": row.split,
            "target_true_vt": int(row.decision == "True"),
            "sampling_rate_hz": header.fs,
            "samples": header.sig_len,
            "channel_count": header.n_sig,
            "channel_names": json.dumps(header.sig_name),
            "channel_units": json.dumps(header.units),
            "ecg_candidates": json.dumps(available_ecg),
            "preferred_ecg": chosen_ecg,
            "preferred_ecg_unit": (
                header.units[header.sig_name.index(chosen_ecg)]
                if chosen_ecg else None
            ),
            "has_pleth_header": ppg_index is not None,
            "pleth_unit": header.units[ppg_index] if ppg_index is not None else None,
            "has_abp_header": "ABP" in header.sig_name,
            "header_member": member,
            "header_sha256": digest,
            "signal_members": json.dumps(signal_members),
        })

        if number % 100 == 0 or number == len(development):
            elapsed = time.monotonic() - started
            eta = elapsed / number * (len(development) - number)
            print(
                f"{number}/{len(development)} headers | "
                f"elapsed {elapsed:.1f}s | estimated remaining {eta:.1f}s",
                flush=True
            )

manifest = pd.DataFrame(manifest_rows)
manifest["ecg_and_pleth_header"] = (
    manifest["preferred_ecg"].notna() & manifest["has_pleth_header"]
)
output = RUN_DIR / "vtac_development_header_manifest.csv"
payload = manifest.to_csv(index=False).encode("utf-8")

if output.exists():
    assert output.read_bytes() == payload, (
         "Existing header manifest differs; stop for review."
    )
output.write_bytes(payload)

provenance = {
    "manifest_sha256": hashlib.sha256(payload).hexdigest(),
    "publisher_checksum_manifest_sha256": hashlib.sha256(checksum_bytes).hexdigest(),
    "labels_sha256": hashlib.sha256(labels_payload).hexdigest(),
    "official_split_sha256": hashlib.sha256(splits_payload).hexdigest(),
    "ecg_preference": ECG_PREFERENCE,
    "header_hashes_verified": len(manifest),
    "waveform_hashes_verified_by_this_cell": 0,
    "test_headers_read": 0,
}
(RUN_DIR / "vtac_header_manifest_provenance.json").write_text(
    json.dumps(provenance, indent=2), encoding="utf-8"
)

print("\nVTAC DEVELOPMENT CHANNEL SUMMARY")
print(manifest.groupby("split").agg(
    events=("event_id", "size"),
    true_vt=("target_true_vt", "sum"),
    ecg_and_pleth=("ecg_and_pleth_header", "sum"),
    abp_present=("has_abp_header", "sum"),
).to_string())

print("\nPreferred ECG counts:")
print(manifest["preferred_ecg"].fillna("NONE").value_counts().to_string())
print("\nPLETH unit counts:")
print(manifest["pleth_unit"].fillna("NO_PLETH").value_counts().to_string())
print("\nChannel-count distribution:")
print(manifest["channel_count"].value_counts().sort_index().to_string())

print("\nHeader presence does not prove a signal contains usable samples.")
print("Test records accessed: 0")
print("Saved:", output)
print(f"Completed in {time.monotonic()-started:.1f}s.")

In [ ]:
# CELL 17 — resumable, publisher-verified VTaC development waveform cache.
# Initial ETA: ~5–30 minutes. Native 250 Hz; no training or test records.
from pathlib import Path, PurePosixPath
from collections import Counter
import hashlib, json, tempfile, time, zipfile
import numpy as np
import pandas as pd
import wfdb

started = time.monotonic()
print("Building VTaC waveform cache — initial ETA ~5–30 minutes.", flush=True)

manifest_path = RUN_DIR / "vtac_development_header_manifest.csv"
provenance = json.loads(
    (RUN_DIR / "vtac_header_manifest_provenance.json").read_text()
)
manifest_hash = hashlib.sha256(manifest_path.read_bytes()).hexdigest()
assert manifest_hash == provenance["manifest_sha256"]

manifest = pd.read_csv(
    manifest_path, dtype={"record_id": str, "event_id": str},
    keep_default_na=False
).sort_values(["split", "record_id", "event_id"]).reset_index(drop=True)

assert len(manifest) == 4555
assert set(manifest["split"]) == {"train", "validation"}
assert not manifest.duplicated(["record_id", "event_id"]).any()

CACHE_DIR = RUN_DIR / "vtac_native250_cache_v1"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 200
policy = {
    "version": "native250-masked-v1",
    "manifest_sha256": manifest_hash,
    "interval_seconds": [290, 300],
    "sampling_rate_hz": 250,
    "channel_order": ["ECG", "PLETH"],
    "ecg_selection": "First candidate with any finite samples; fixed header preference.",
    "missing_storage": "Zero placeholder with separate sample-validity mask.",
    "constant_signals": "Retained.",
    "batch_size": BATCH_SIZE,
}
policy_path = CACHE_DIR / "policy.json"
if policy_path.exists():
    assert json.loads(policy_path.read_text()) == policy, "Cache policy changed."
else:
    policy_path.write_text(json.dumps(policy, indent=2), encoding="utf-8")

def file_hash(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()

summaries = []
processed_this_run = 0

with zipfile.ZipFile(ARCHIVES["vtac-1.1"]) as archive:
    checksum_names = [
        n for n in archive.namelist()
        if PurePosixPath(n).name == "SHA256SUMS.txt"
    ]
    assert len(checksum_names) == 1
    checksum_name = checksum_names[0]
    archive_root = str(PurePosixPath(checksum_name).parent)
    checksum_bytes = archive.read(checksum_name)
    assert hashlib.sha256(checksum_bytes).hexdigest() == (
        provenance["publisher_checksum_manifest_sha256"]
    )

    hashes = {}
    for line in checksum_bytes.decode("utf-8-sig").splitlines():
        if not line.strip():
            continue
        digest, name = line.split(maxsplit=1)
        name = name.lstrip("*")
        if name.startswith("./"):
            name = name[2:]
        assert name not in hashes
        hashes[name] = digest.lower()

    def verified_read(member):
        relative = str(PurePosixPath(member).relative_to(archive_root))
        expected = {hashes[k] for k in (relative, member) if k in hashes}
        assert len(expected) == 1, f"Missing/conflicting hash: {member}"
        payload = archive.read(member)
        assert hashlib.sha256(payload).hexdigest() == next(iter(expected)), (
            f"Publisher checksum failed: {member}"
        )
        return payload

    for batch_number, start in enumerate(
        range(0, len(manifest), BATCH_SIZE)
    ):
        batch = manifest.iloc[start:start + BATCH_SIZE]
        shard = CACHE_DIR / f"batch_{batch_number:03d}.npz"
        receipt = CACHE_DIR / f"batch_{batch_number:03d}.json"
        expected_ids = batch["event_id"].tolist()

        if shard.exists() and receipt.exists():
            saved = json.loads(receipt.read_text())
            assert saved["event_ids"] == expected_ids
            assert saved["manifest_sha256"] == manifest_hash
            assert file_hash(shard) == saved["shard_sha256"]
            summaries.extend(saved["event_summaries"])
            print(f"Reused verified batch {batch_number:03d}.", flush=True)
            continue

        signals, masks, event_summaries = [], [], []

        for row in batch.itertuples(index=False):
            # Temporary source files are removed after each event.
            with tempfile.TemporaryDirectory(
                prefix="vtac-read-", dir="/content"
            ) as temporary:
                folder = Path(temporary)
                assert Path(row.event_id).name == row.event_id

                header_bytes = verified_read(row.header_member)
                assert hashlib.sha256(header_bytes).hexdigest() == row.header_sha256
                (folder / f"{row.event_id}.hea").write_bytes(header_bytes)
                header = wfdb.rdheader(str(folder / row.event_id))
                assert header.fs == 250 and header.sig_len == 90000

                parent = PurePosixPath(row.header_member).parent
                for filename in sorted(set(header.file_name)):
                    assert PurePosixPath(filename).name == filename
                    assert "/" not in filename and "\\" not in filename
                    payload = verified_read(str(parent / filename))
                    (folder / filename).write_bytes(payload)

                record = wfdb.rdrecord(
                    str(folder / row.event_id),
                    sampfrom=72500, sampto=75000, physical=True
                )
                values = record.p_signal
                assert values.shape == (2500, header.n_sig)

                ecg_name = None
                for candidate in json.loads(row.ecg_candidates):
                    index = record.sig_name.index(candidate)
                    if np.isfinite(values[:, index]).any():
                        ecg_name = candidate
                        break

                chosen = [
                    ecg_name,
                    "PLETH" if "PLETH" in record.sig_name else None
                ]
                x = np.zeros((2, 2500), dtype=np.float32)
                mask = np.zeros((2, 2500), dtype=bool)
                constant = [False, False]

                for channel, name in enumerate(chosen):
                    if name is None:
                        continue
                    index = record.sig_name.index(name)
                    if channel == 0:
                        assert record.units[index] == "mV", (
                            f"Unexpected ECG unit in {row.event_id}"
                        )
                    source_values = values[:, index]
                    valid_samples = np.isfinite(source_values)
                    mask[channel] = valid_samples
                    x[channel, valid_samples] = source_values[valid_samples]
                    if valid_samples.any():
                        constant[channel] = bool(
                            np.ptp(source_values[valid_samples]) == 0
                        )

                assert np.isfinite(x).all(), "Float conversion overflow."
                signals.append(x)
                masks.append(mask)

                event_summaries.append({
                    "event_id": row.event_id,
                    "record_id": row.record_id,
                    "split": row.split,
                    "target_true_vt": int(row.target_true_vt),
                    "selected_ecg": ecg_name,
                    "ecg_valid_samples": int(mask[0].sum()),
                    "ppg_valid_samples": int(mask[1].sum()),
                    "ecg_constant": constant[0],
                    "ppg_constant": constant[1],
                })

            processed_this_run += 1
            if processed_this_run % 25 == 0:
                elapsed = time.monotonic() - started
                completed = start + len(event_summaries)
                remaining = len(manifest) - completed
                eta = elapsed / processed_this_run * remaining
                print(
                    f"{completed}/4555 events | elapsed {elapsed/60:.1f} min | "
                    f"estimated remaining {eta/60:.1f} min",
                    flush=True
                )

        # Write the shard first; its receipt marks a completed batch.
        partial = shard.with_suffix(".partial")
        with partial.open("wb") as stream:
            np.savez_compressed(
                stream,
                x=np.stack(signals),                 # [N, 2, 2500]
                valid=np.stack(masks),               # same shape
                y=batch["target_true_vt"].to_numpy(dtype=np.int64),
                event_id=batch["event_id"].to_numpy(dtype=str),
                record_id=batch["record_id"].to_numpy(dtype=str),
                split=batch["split"].to_numpy(dtype=str),
            )
        partial.replace(shard)

        saved = {
            "manifest_sha256": manifest_hash,
            "event_ids": expected_ids,
            "shard_sha256": file_hash(shard),
            "event_summaries": event_summaries,
        }
        receipt_partial = receipt.with_suffix(".partial")
        receipt_partial.write_text(json.dumps(saved, indent=2), encoding="utf-8")
        receipt_partial.replace(receipt)

        summaries.extend(event_summaries)
        print(
            f"Saved batch {batch_number:03d}: {len(batch)} events.",
            flush=True
        )

summary = pd.DataFrame(summaries)
assert len(summary) == 4555 and summary["event_id"].is_unique
summary.to_csv(CACHE_DIR / "event_quality_manifest.csv", index=False)

summary["ecg_absent"] = summary["ecg_valid_samples"].eq(0)
summary["ppg_absent"] = summary["ppg_valid_samples"].eq(0)
summary["ecg_partial"] = summary["ecg_valid_samples"].between(1, 2499)
summary["ppg_partial"] = summary["ppg_valid_samples"].between(1, 2499)

print("\nVTAC WAVEFORM CACHE SUMMARY")
print(summary.groupby("split").agg(
    events=("event_id", "size"),
    true_vt=("target_true_vt", "sum"),
    ecg_absent=("ecg_absent", "sum"),
    ppg_absent=("ppg_absent", "sum"),
    ecg_partial=("ecg_partial", "sum"),
    ppg_partial=("ppg_partial", "sum"),
    ecg_constant=("ecg_constant", "sum"),
    ppg_constant=("ppg_constant", "sum"),
).to_string())
print("\nCache:", CACHE_DIR)
print("Native input shape per event: [2 channels, 2500 samples].")
print("Missing samples require the valid mask; zeros alone are ambiguous.")
print("Test events accessed: 0")
print(f"Completed in {(time.monotonic()-started)/60:.1f} minutes.")

In [ ]:
# CELL 18 — restore saved cache, verify it, and prepare masked 125-Hz inputs.
# ETA: ~20–90 seconds, excluding Drive authorisation. No model training.
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")

import hashlib, json, time
import numpy as np
import pandas as pd
from scipy.signal import resample_poly
from scipy.ndimage import minimum_filter1d
import scipy
import torch

started = time.monotonic()
print("Preparing M2 inputs — initial ETA ~20–90s.", flush=True)

RUN_DIR = Path(
    "/content/drive/MyDrive/SentinelWard-work/data-audit/"
    "20261015T132438_754996Z"
)
CACHE_DIR = RUN_DIR / "vtac_native250_cache_v1"
PREP_DIR = RUN_DIR / "vtac_m2_prepared_v1"
PREP_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device available:", device, flush=True)
if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0), flush=True)
else:
    print("Preparation can proceed on CPU; GPU is recommended for training.")

def sha256_file(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024**2), b""):
            h.update(block)
    return h.hexdigest()

manifest_path = RUN_DIR / "vtac_development_header_manifest.csv"
policy = json.loads((CACHE_DIR / "policy.json").read_text())
assert sha256_file(manifest_path) == policy["manifest_sha256"]
assert policy["channel_order"] == ["ECG", "PLETH"]
assert policy["sampling_rate_hz"] == 250
assert policy["interval_seconds"] == [290, 300]

manifest = pd.read_csv(
    manifest_path,
    dtype={"record_id": str, "event_id": str},
    keep_default_na=False,
).sort_values(["split", "record_id", "event_id"]).reset_index(drop=True)

parts, targets, event_ids, record_ids, split_names = [], [], [], [], []
source_shards = {}
receipts = sorted(CACHE_DIR.glob("batch_*.json"))
assert len(receipts) == 23, f"Expected 23 completed batches; found {len(receipts)}"

for number, receipt_path in enumerate(receipts, start=1):
    receipt = json.loads(receipt_path.read_text())
    shard = receipt_path.with_suffix(".npz")
    assert receipt["manifest_sha256"] == policy["manifest_sha256"]
    digest = sha256_file(shard)
    assert digest == receipt["shard_sha256"], f"Cache changed: {shard.name}"
    source_shards[shard.name] = digest

    with np.load(shard, allow_pickle=False) as stored:
        x = stored["x"].astype(np.float64)
        valid = stored["valid"].astype(bool)
        assert x.shape == valid.shape
        assert x.shape[1:] == (2, 2500)
        assert np.isfinite(x).all()
        assert stored["event_id"].tolist() == receipt["event_ids"]

        # Per-event, per-channel statistics from valid pre-alarm samples only.
        count = valid.sum(axis=-1, keepdims=True)
        denominator = np.maximum(count, 1)
        mean = np.where(valid, x, 0).sum(axis=-1, keepdims=True) / denominator
        centred = np.where(valid, x - mean, 0)
        variance = (centred ** 2).sum(axis=-1, keepdims=True) / denominator
        scale = np.maximum(np.sqrt(variance), 1e-6)
        normalised = centred / scale

        # Explicit anti-alias FIR filter: 41 input samples, +/-20 support.
        # No samples at or after the alarm are used.
        from scipy.signal import firwin
        taps = firwin(41, cutoff=0.5, window=("kaiser", 5.0))
        downsampled = resample_poly(
            normalised, up=1, down=2, axis=-1,
            window=taps, padtype="constant"
        )

        # Conservatively invalidate outputs near gaps and window boundaries.
        safe_native = minimum_filter1d(
            valid.astype(np.uint8), size=41,
            axis=-1, mode="constant", cval=0
        ).astype(bool)
        safe_125 = safe_native[..., ::2]
        assert downsampled.shape == safe_125.shape

        downsampled = np.where(safe_125, downsampled, 0)
        prepared = np.concatenate(
            [downsampled, safe_125.astype(np.float64)], axis=1
        ).astype(np.float32)
        assert prepared.shape[1:] == (4, 1250)
        assert np.isfinite(prepared).all()

        parts.append(prepared)
        targets.append(stored["y"].copy())
        event_ids.append(stored["event_id"].copy())
        record_ids.append(stored["record_id"].copy())
        split_names.append(stored["split"].copy())

    elapsed = time.monotonic() - started
    eta = elapsed / number * (len(receipts) - number)
    print(
        f"Batch {number}/23 | elapsed {elapsed:.1f}s | "
        f"estimated remaining {eta:.1f}s",
        flush=True
    )

X = np.concatenate(parts)
y = np.concatenate(targets).astype(np.int64)
events = np.concatenate(event_ids)
records = np.concatenate(record_ids)
splits = np.concatenate(split_names)
del parts

assert X.shape == (4555, 4, 1250)
assert len(set(events.tolist())) == 4555
assert events.tolist() == manifest["event_id"].tolist()
assert records.tolist() == manifest["record_id"].tolist()
assert splits.tolist() == manifest["split"].tolist()
np.testing.assert_array_equal(y, manifest["target_true_vt"].to_numpy())
assert set(splits) == {"train", "validation"}
assert not set(records[splits == "train"]) & set(records[splits == "validation"])

# Do not silently train on an event with no usable input at all.
usable_ecg = X[:, 2, :].sum(axis=1) > 0
usable_ppg = X[:, 3, :].sum(axis=1) > 0
assert usable_ecg.all(), (
    f"{int((~usable_ecg).sum())} events lost all ECG support; review before training."
)

configuration = {
    "version": "m2-preprocessing-v1",
    "source_manifest_sha256": policy["manifest_sha256"],
    "source_shard_sha256": source_shards,
    "channel_order": ["ecg_z", "ppg_z", "ecg_valid", "ppg_valid"],
    "input_shape": [4, 1250],
    "interval_seconds": [290, 300],
    "normalisation": "Per-window valid-sample mean/std; minimum scale 1e-6.",
    "resampling": "250 to 125 Hz; explicit 41-tap Kaiser FIR, cutoff 0.5 Nyquist.",
    "mask": "41-sample conservative validity erosion, then decimation.",
    "scipy_version": scipy.__version__,
    "numpy_version": np.__version__,
}
config_path = PREP_DIR / "preprocessing.json"
if config_path.exists():
    assert json.loads(config_path.read_text()) == configuration, (
        "Existing preprocessing configuration differs; stop for review."
    )

output = PREP_DIR / "development.npz"
partial = PREP_DIR / "development.partial"
with partial.open("wb") as stream:
    np.savez_compressed(
        stream, X=X, y=y, event_id=events, record_id=records, split=splits
    )
partial.replace(output)
config_path.write_text(json.dumps(configuration, indent=2), encoding="utf-8")
(PREP_DIR / "development.sha256").write_text(
    sha256_file(output), encoding="utf-8"
)

summary = pd.DataFrame({
    "split": splits,
    "true_vt": y,
    "usable_ecg": usable_ecg,
    "usable_ppg": usable_ppg,
})
print("\nM2 INPUT SUMMARY")
print(summary.groupby("split").agg(
    events=("true_vt", "size"),
    true_vt=("true_vt", "sum"),
    usable_ecg=("usable_ecg", "sum"),
    usable_ppg=("usable_ppg", "sum"),
).to_string())
print("Input shape:", X.shape)
print("Device:", device)
print("Test events processed: 0")
print("Saved:", output)
print(f"Completed in {time.monotonic()-started:.1f}s.")